In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow_addons as tfa
import os

####################################
#Multi_gpu usage strategy run code.#
####################################

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        print(gpus[0])
        print(gpus[1])
        print(len(gpus), "Physical GPUs,", len(gpus), "Logical GPUs")
    except RuntimeError as e:
        print(e)  
else:
    print("No GPUs found")

strategy = tf.distribute.MirroredStrategy(devices=["/GPU:0", "/GPU:1"])
print(f'Number of replicas: {strategy.num_replicas_in_sync}')

In [ ]:
print(tf.__version__)

IMG_H, IMG_W = 128, 128
LATENT_DIM = 128 
CHANNELS_TERRAIN = 3
CHANNELS_FIRE = 1
# 기상 데이터: 온습도와 풍력 각각 (4,2)
WEATHER_TNH_SHAPE = (4,2)
WEATHER_WIND_SHAPE = (4,2)
# 학습 데이터셋: fire_seq는 4개 프레임 (0h, 4h, 8h, 12h), target_seq는 3개 (0→4,4→8,8→12)
TIME_STEPS_MAX = 3

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, Conv2DTranspose, Dense, Reshape, 
                                     Concatenate, Lambda, MaxPooling2D, GlobalAveragePooling2D,UpSampling2D,BatchNormalization,Subtract,
                                     Flatten, ReLU, ReLU ,Dropout, Add, Multiply)
from tensorflow.keras.models import Model
from tensorflow.keras.layers.experimental.preprocessing import Resizing
import tensorflow.keras.backend as K
from tensorflow.keras import layers as L

# ======================================================
# 상수 설정
# ======================================================
IMG_H, IMG_W = 128, 128
LATENT_DIM = 128 
CHANNELS_TERRAIN = 3
CHANNELS_FIRE = 1
# 기상 데이터: 온습도와 풍력 각각 (4,2)
WEATHER_TNH_SHAPE = (4,2)
WEATHER_WIND_SHAPE = (4,2)
# 학습 데이터셋: fire_seq는 4개 프레임 (0h, 4h, 8h, 12h), target_seq는 3개 (0→4,4→8,8→12)
TIME_STEPS_MAX = 3

def film_block(feature_map, cond_vector, name=None):
    """
    feature_map: (B, H, W, C) — Tensor
    cond_vector: (B, K)      — Tensor
    """
    # 1) static 채널 수 가져오기
    C = feature_map.shape[-1]    # Python int 로 반환됩니다.
    # 2) scale (gamma) 생성
    gamma = Dense(C, name=f'{name}_gamma')(cond_vector)  # (B, C)
    gamma = Reshape((1, 1, C))(gamma)                    # (B,1,1,C)
    # 3) shift (beta) 생성
    beta  = Dense(C, name=f'{name}_beta')(cond_vector)   # (B, C)
    beta  = Reshape((1, 1, C))(beta)                      # (B,1,1,C)
    # 4) spatial broadcast 및 적용
    return feature_map * gamma + beta

# Define a basic residual block for both downsampling and upsampling
def residual_block(x, filters, downsample=False, upsample=False):
    identity = x

    if downsample:
        # Downsample identity to match the output shape
        identity = Conv2D(filters, (1, 1), strides=(2, 2), padding='same')(identity)
        x = Conv2D(filters, (3, 3), strides=(2, 2), padding='same')(x)
    elif upsample:
        # Upsample identity to match the output shape
        identity = Conv2DTranspose(filters, (1, 1), strides=(2, 2), padding='same')(identity)
        x = Conv2DTranspose(filters, (3, 3), strides=(2, 2), padding='same')(x)
    else:
        # Adjust the identity tensor if filters don't match
        if identity.shape[-1] != filters:
            identity = Conv2D(filters, (1, 1), padding='same')(identity)
        
        x = Conv2D(filters, (3, 3), padding='same')(x)

    x = BatchNormalization()(x)
    x = tf.keras.activations.swish(x)
    
    if not upsample:
        x = Conv2D(filters, (3, 3), padding='same')(x)
    else:
        x = Conv2DTranspose(filters, (3, 3), padding='same')(x)
        
    x = BatchNormalization()(x)
    
    # Residual connection
    x = Add()([x, identity])
    x = tf.keras.activations.swish(x)
    return x

# Define the Backbone Net like an Autoencoder with Residual Blocks
def BackboneNet(input_tensor, noise_tensor):
    # Encoding part (downsampling)
    x = Conv2D(64, (3, 3), padding='same')(input_tensor) # 채널 수를 64로 시작하는 것을 추천
    x = BatchNormalization()(x)
    x = tf.keras.activations.swish(x)
    s1 = x  # Skip Connection 1 저장 (128x128)

    # Downsampling with residual blocks
    x = residual_block(x, 64, downsample=True) # 64x64
    s2 = x  # Skip Connection 2 저장

    x = residual_block(x, 128, downsample=True) # 32x32
    s3 = x  # Skip Connection 3 저장

    x = residual_block(x, 256, downsample=True) # 16x16

    # Bottleneck layer
    x = residual_block(x, 512)
    x = Concatenate()([x, noise_tensor]) # 노이즈 주입

    # Decoding part (upsampling)
    x = residual_block(x, 256, upsample=True) # 32x32
    x = Concatenate()([x, s3]) # Skip Connection 3 연결

    x = residual_block(x, 128, upsample=True) # 64x64
    x = Concatenate()([x, s2]) # Skip Connection 2 연결

    x = residual_block(x, 64, upsample=True) # 128x128
    x = Concatenate()([x, s1]) # Skip Connection 1 연결
    
    # 마지막 레이어는 채널 수를 부드럽게 줄여주는 것이 좋습니다.
    x = Conv2D(16, (3, 3), padding='same', activation=tf.keras.activations.swish)(x)
    
    # 최종 출력
    output = Conv2D(1, (3, 3), padding='same', activation='tanh')(x)
    
    return output


    
# ======================================
# Generator 모델: 모든 모달리티를 full resolution에서 처리 후 U-Net 구조에 통합
# ======================================

# --- util ---
def silu(x):
    return tf.nn.silu(x)

def build_generator():
    # =========================================================
    # Inputs
    # =========================================================
    terrain   = L.Input(shape=(128,128,3),  name="terrain")
    tnh_in    = L.Input(shape=(4,2),        name="tnh")
    wind_in   = L.Input(shape=(4,2),        name="wind")
    current   = L.Input(shape=(128,128,1),  name="current_fire")
    step_n    = L.Input(shape=(3,),         name="step_n")
    noise_in  = L.Input(shape=(128,),       name="noise")

    # =========================================================
    # Terrain stem
    # =========================================================
    x_t = L.Conv2D(32, 3, padding="same", name="conv2d_686")(terrain)
    x_t = L.BatchNormalization(name="batch_normalization_644")(x_t)
    x_t = L.Lambda(silu, name="tf.nn.silu_558")(x_t)

    x_t = L.Conv2D(64, 3, padding="same", name="conv2d_687")(x_t)
    x_t = L.BatchNormalization(name="batch_normalization_645")(x_t)
    x_t = L.Lambda(silu, name="tf.nn.silu_559")(x_t)

    # =========================================================
    # Weather FiLM (tnh, wind)  — (gamma·x + beta)
    # =========================================================
    tnh_flat   = L.Flatten(name="flatten_108")(tnh_in)       # (None,8)
    wind_flat  = L.Flatten(name="flatten_109")(wind_in)      # (None,8)

    tnh_gamma  = L.Dense(64, name="film_tnh_gamma")(tnh_flat)
    tnh_beta   = L.Dense(64, name="film_tnh_beta")(tnh_flat)
    wind_gamma = L.Dense(64, name="film_wind_gamma")(wind_flat)
    wind_beta  = L.Dense(64, name="film_wind_beta")(wind_flat)

    tnh_g  = L.Reshape((1,1,64), name="reshape_343")(tnh_gamma)
    tnh_b  = L.Reshape((1,1,64), name="reshape_344")(tnh_beta)
    wind_g = L.Reshape((1,1,64), name="reshape_345")(wind_gamma)
    wind_b = L.Reshape((1,1,64), name="reshape_346")(wind_beta)

    x_t_tnh  = L.Add(name="tf.__operators__.add_162")([
                    L.Multiply(name="tf.math.multiply_162")([x_t, tnh_g]), tnh_b])
    x_t_wind = L.Add(name="tf.__operators__.add_163")([
                    L.Multiply(name="tf.math.multiply_163")([x_t, wind_g]), wind_b])

    x_t_film = L.Add(name="add_233")([x_t_tnh, x_t_wind])     # (128,128,64)

    # =========================================================
    # Current fire + step FiLM
    # =========================================================
    x_c = L.Conv2D(64, 3, padding="same", name="conv2d_688")(current)
    x_c = L.BatchNormalization(name="batch_normalization_646")(x_c)
    x_c = L.Lambda(silu, name="tf.nn.silu_560")(x_c)

    step_gamma = L.Dense(64, name="film_step_gamma")(step_n)
    step_beta  = L.Dense(64, name="film_step_beta")(step_n)
    step_g     = L.Reshape((1,1,64), name="reshape_347")(step_gamma)
    step_b     = L.Reshape((1,1,64), name="reshape_348")(step_beta)

    x_c_film = L.Add(name="tf.__operators__.add_164")([
                    L.Multiply(name="tf.math.multiply_164")([x_c, step_g]), step_b])

    # =========================================================
    # Fuse @ 128×128
    # =========================================================
    x = L.Concatenate(name="concatenate_76")([x_t_film, x_c_film])    # (128,128,128)
    x = L.Conv2D(64, 3, padding="same", name="conv2d_689")(x)
    x = L.BatchNormalization(name="batch_normalization_647")(x)
    x = L.Lambda(silu, name="tf.nn.silu_561")(x)
    enc_128 = x

    # =========================================================
    # Down 1: 128→64
    # =========================================================
    y  = L.Conv2D(64, 3, strides=2, padding="same", name="conv2d_691")(x)
    y  = L.BatchNormalization(name="batch_normalization_648")(y)
    y  = L.Lambda(silu, name="tf.nn.silu_562")(y)
    y  = L.Conv2D(64, 3, padding="same", name="conv2d_692")(y)
    y  = L.BatchNormalization(name="batch_normalization_649")(y)

    sc1 = L.Conv2D(64, 1, strides=2, padding="same", name="conv2d_690")(x)
    y   = L.Add(name="add_234")([y, sc1])
    y   = L.Lambda(silu, name="tf.nn.silu_563")(y)
    enc_64 = y

    # =========================================================
    # Down 2: 64→32
    # =========================================================
    y2  = L.Conv2D(128, 3, strides=2, padding="same", name="conv2d_694")(y)
    y2  = L.BatchNormalization(name="batch_normalization_650")(y2)
    y2  = L.Lambda(silu, name="tf.nn.silu_564")(y2)
    y2  = L.Conv2D(128, 3, padding="same", name="conv2d_695")(y2)
    y2  = L.BatchNormalization(name="batch_normalization_651")(y2)

    sc2 = L.Conv2D(128, 1, strides=2, padding="same", name="conv2d_693")(y)
    y2  = L.Add(name="add_235")([y2, sc2])
    y2  = L.Lambda(silu, name="tf.nn.silu_565")(y2)
    enc_32 = y2

    # =========================================================
    # Down 3: 32→16
    # =========================================================
    y3  = L.Conv2D(256, 3, strides=2, padding="same", name="conv2d_697")(y2)
    y3  = L.BatchNormalization(name="batch_normalization_652")(y3)
    y3  = L.Lambda(silu, name="tf.nn.silu_566")(y3)
    y3  = L.Conv2D(256, 3, padding="same", name="conv2d_698")(y3)
    y3  = L.BatchNormalization(name="batch_normalization_653")(y3)

    # (FIXED: stride=2 to match 16×16)
    sc3 = L.Conv2D(256, 1, strides=2, padding="same", name="conv2d_696")(y2)
    y3  = L.Add(name="add_236")([y3, sc3])
    y3  = L.Lambda(silu, name="tf.nn.silu_567")(y3)

    # =========================================================
    # Bottleneck @16×16 (Residual 512ch)
    # =========================================================
    y4  = L.Conv2D(512, 3, padding="same", name="conv2d_700")(y3)
    y4  = L.BatchNormalization(name="batch_normalization_654")(y4)
    y4  = L.Lambda(silu, name="tf.nn.silu_568")(y4)
    y4  = L.Conv2D(512, 3, padding="same", name="conv2d_701")(y4)
    y4  = L.BatchNormalization(name="batch_normalization_655")(y4)

    sc4 = L.Conv2D(512, 1, padding="same", name="conv2d_699")(y3)
    y4  = L.Add(name="add_237")([y4, sc4])
    y4  = L.Lambda(silu, name="tf.nn.silu_569")(y4)

    # =========================================================
    # Noise 128 → 4096 → (16,16,16) & concat
    # =========================================================
    z   = L.Dense(4096, name="dense_27")(noise_in)
    z   = L.Reshape((16,16,16), name="reshape_349")(z)
    dec_in = L.Concatenate(name="concatenate_77")([y4, z])            # (16,16,528)

    # =========================================================
    # Up 1: 16→32 (Residual Deconv) + skip(enc_32)
    # =========================================================
    u   = L.Conv2DTranspose(256, 3, strides=2, padding="same", name="conv2d_transpose_200")(dec_in)
    u   = L.BatchNormalization(name="batch_normalization_656")(u)
    u   = L.Lambda(silu, name="tf.nn.silu_570")(u)
    u   = L.Conv2DTranspose(256, 3, padding="same", name="conv2d_transpose_201")(u)
    u   = L.BatchNormalization(name="batch_normalization_657")(u)

    u_sc= L.Conv2DTranspose(256, 1, strides=2, padding="same", name="conv2d_transpose_199")(dec_in)
    u   = L.Add(name="add_238")([u, u_sc])
    u   = L.Lambda(silu, name="tf.nn.silu_571")(u)
    u   = L.Concatenate(name="concatenate_78")([u, enc_32])           # (32,32,384)

    # =========================================================
    # Up 2: 32→64 (Residual Deconv) + skip(enc_64)
    # =========================================================
    u2  = L.Conv2DTranspose(128, 3, strides=2, padding="same", name="conv2d_transpose_203")(u)
    u2  = L.BatchNormalization(name="batch_normalization_658")(u2)
    u2  = L.Lambda(silu, name="tf.nn.silu_572")(u2)
    u2  = L.Conv2DTranspose(128, 3, padding="same", name="conv2d_transpose_204")(u2)
    u2  = L.BatchNormalization(name="batch_normalization_659")(u2)

    u2_sc = L.Conv2DTranspose(128, 1, strides=2, padding="same", name="conv2d_transpose_202")(u)
    u2    = L.Add(name="add_239")([u2, u2_sc])
    u2    = L.Lambda(silu, name="tf.nn.silu_573")(u2)
    u2    = L.Concatenate(name="concatenate_79")([u2, enc_64])        # (64,64,192)

    # =========================================================
    # Up 3: 64→128 (Residual Deconv) + skip(enc_128)
    # =========================================================
    u3  = L.Conv2DTranspose(64, 3, strides=2, padding="same", name="conv2d_transpose_206")(u2)
    u3  = L.BatchNormalization(name="batch_normalization_660")(u3)
    u3  = L.Lambda(silu, name="tf.nn.silu_574")(u3)
    u3  = L.Conv2DTranspose(64, 3, padding="same", name="conv2d_transpose_207")(u3)
    u3  = L.BatchNormalization(name="batch_normalization_661")(u3)

    u3_sc = L.Conv2DTranspose(64, 1, strides=2, padding="same", name="conv2d_transpose_205")(u2)
    u3    = L.Add(name="add_240")([u3, u3_sc])
    u3    = L.Lambda(silu, name="tf.nn.silu_575")(u3)
    u3    = L.Concatenate(name="concatenate_80")([u3, enc_128])       # (128,128,128)

    # =========================================================
    # Heads
    # =========================================================
    out_mid = L.Conv2D(32, 3, padding="same", name="conv2d_702")(u3)
    out_img = L.Conv2D(1,  3, padding="same", activation="sigmoid", name="conv2d_703")(out_mid)

    return Model(
        inputs=[current, terrain, tnh_in, wind_in , step_n, noise_in],
        outputs=out_img,
        name="Generator",
    )

# ======================================
# Discriminator 모델 (PatchGAN 스타일, weather도 full resolution 처리)
# ======================================
def build_discriminator():
    
    # --- 입력 레이어 ---
    current_fire_inp = Input(shape=(IMG_H, IMG_W, CHANNELS_FIRE), name="current_fire")
    terrain_inp      = Input(shape=(IMG_H, IMG_W, CHANNELS_TERRAIN), name="terrain")
    tnh_inp          = Input(shape=WEATHER_TNH_SHAPE, name="tnh")
    wind_inp         = Input(shape=WEATHER_WIND_SHAPE, name="wind")
    step_inp         = Input(shape=(3), name="step_n")

    
    # --- Terrain Feature 추출 ---
    # 해상도 유지: stride=1로 conv 진행하여 (IMG_H,IMG_W,64) feature map을 생성
    terrain_feature = Conv2D(32, kernel_size=3, strides=1, padding='same')(terrain_inp)
    terrain_feature = BatchNormalization()(terrain_feature)
    terrain_feature = tf.keras.activations.swish(terrain_feature)
    terrain_feature = Conv2D(64, kernel_size=3, strides=1, padding='same')(terrain_feature)
    terrain_feature = BatchNormalization()(terrain_feature)
    terrain_feature = tf.keras.activations.swish(terrain_feature)
    # terrain_feature: (192,192,64)
     # FiLM-modulated terrain
    tnh_flat = Flatten()(tnh_inp)
    wind_flat= Flatten()(wind_inp)
    t_tnh_mod  = film_block(terrain_feature, tnh_flat,  name='film_tnh')
    t_wind_mod= film_block(terrain_feature, wind_flat, name='film_wind')
    terrain_effect = Add()([t_tnh_mod, t_wind_mod])  # (B,H,W,64)

    # Fire feature
    ffeat = Conv2D(64,3,padding='same')(current_fire_inp)
    ffeat = BatchNormalization()(ffeat)
    ffeat = tf.keras.activations.swish(ffeat)

    # Step modulation (same FiLM)
    #step_feat = Dense(64,activation='swish')(step_inp)
    ffeat = film_block(ffeat, step_inp, name='film_step')

    fused = Concatenate()([terrain_effect, ffeat])
    
    # --- Convolutional Layers ---
    # PatchGAN 스타일로 패치별 판별을 수행하기 위해, 여러 Conv 레이어로 spatial dimension을 줄여나감

    x = Conv2D(32, kernel_size=3, strides=2, padding='same')(fused)
    
    x = tf.keras.activations.swish(x)  # (96,96,64)
    x = Conv2D(64, kernel_size=3, strides=2, padding='same')(x)
    
    x = tf.keras.activations.swish(x)  # (96,96,64)
    
    x = Conv2D(128, kernel_size=3, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.swish(x)  # (48,48,128)
    
    x = Conv2D(256, kernel_size=3, strides=2, padding='same')(x)
    
    x = tf.keras.activations.swish(x)  # (24,24,256)
    
    # 마지막 conv layer는 stride 1을 사용하여 마지막 패치의 출력을 생성합니다.
    x = Conv2D(512, kernel_size=3, strides=2, padding='same')(x)

    x = tf.keras.activations.swish(x)  # (24,24,512)
    #x = Flatten()(x)
    #output = Dense(1, activation='sigmoid')(x)
    # 최종 출력: 각 패치에 대해 진위(logit)를 반환 (패치당 하나의 스칼라 값)
    output = Conv2D(1, kernel_size=3,  strides=1, padding='same')(x)
    # 출력 shape은 (24,24,1)로, 각 위치가 하나의 패치에 대한 판단값을 의미합니다.
    
    model = Model(inputs=[current_fire_inp, terrain_inp, tnh_inp, wind_inp, step_inp], outputs=output, name="Discriminator")
    return model


# ======================================
# ProgressiveStepGAN: 단일 스텝 예측을 위한 학습 (각 학습 쌍은 무작위 선택)
# ======================================
class ProgressiveStepGAN(tf.keras.Model):
    def __init__(self, generator, discriminator,
                 lambda_L1=20.0, lambda_dice=0.5, lambda_gp=10.0, n_critic=4, **kwargs):
        """
        WGAN-GP 버전. lambda_adv는 사용되지 않습니다.
        lambda_gp: 그래디언트 페널티 가중치 (보통 10)
        """
        super(ProgressiveStepGAN, self).__init__(**kwargs)
        self.generator = generator
        self.discriminator = discriminator # ★★★ 마지막에 Sigmoid가 없어야 함 ★★★
        self.lambda_L1 = lambda_L1
        self.lambda_dice = lambda_dice
        self.lambda_gp = lambda_gp
        self.n_critic = n_critic
        # BCE는 더 이상 사용되지 않습니다.
    
    def compile(self, gen_optimizer, disc_optimizer):
        super(ProgressiveStepGAN, self).compile()
        self.gen_optimizer = gen_optimizer
        self.disc_optimizer = disc_optimizer

    # 위에서 정의한 gradient_penalty 함수를 여기에 붙여넣으세요.
    def gradient_penalty(self, real_images, fake_images, disc_inputs):
        """ WGAN-GP for PatchGAN: reduce patch map to scalar per-sample before grads """
        (terrain, selected_tnh, selected_wind, step_indices) = disc_inputs
        batch_size = tf.shape(real_images)[0]

        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0., 1.)
        inter = real_images * alpha + fake_images * (1 - alpha)

        with tf.GradientTape() as tape:
            tape.watch(inter)
            pred_map = self.discriminator([inter, terrain, selected_tnh, selected_wind, step_indices],
                                        training=True)
            # per-sample scalar
            pred = tf.reduce_mean(pred_map, axis=[1,2,3])

        # grads wrt images
        grads = tape.gradient(pred, inter)
        grads = tf.cast(grads, tf.float32)  # AMP 대비
        # ||∇||2 per-sample
        grad_norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2,3]) + 1e-12)
        gp = tf.reduce_mean((grad_norm - 1.0) ** 2)
        return gp
    
    

    def train_step(self, data):
        (terrain, fire_seq, tnh_seq, wind_seq) = data
        batch_size = tf.shape(terrain)[0]
        
        random_indices = tf.random.uniform(shape=(batch_size,), minval=0, maxval=3, dtype=tf.int32)
        step_indices = tf.one_hot(random_indices, depth=TIME_STEPS_MAX, dtype=tf.float32)

        selected_fire = tf.gather(fire_seq, random_indices, axis=1, batch_dims=1)
        selected_target = tf.gather(fire_seq, random_indices + 1, axis=1, batch_dims=1)
        selected_tnh = tf.gather(tnh_seq, random_indices, axis=1, batch_dims=1)
        selected_wind = tf.gather(wind_seq, random_indices, axis=1, batch_dims=1)

        # =======================================================
        # 1. 판별자(Critic) 학습
        # =======================================================
        for _ in range(self.n_critic):
            with tf.GradientTape() as disc_tape:
                noise = tf.random.normal(shape=(batch_size, LATENT_DIM))
                # 생성된 가짜 이미지
                pred = self.generator([selected_fire, terrain, selected_tnh, selected_wind, step_indices, noise], training=False)
                
                # 진짜/가짜 이미지에 대한 판별자 점수
                disc_real_score = self.discriminator([selected_target, terrain, selected_tnh, selected_wind, step_indices], training=True)
                disc_fake_score = self.discriminator([pred, terrain, selected_tnh, selected_wind, step_indices], training=True)

                # 판별자 손실 (WGAN)
                d_loss_wgan = tf.reduce_mean(disc_fake_score) - tf.reduce_mean(disc_real_score)

                # 그래디언트 페널티 계산
                gp = self.gradient_penalty(selected_target, pred, (terrain, selected_tnh, selected_wind, step_indices))
                
                # 최종 판별자 손실
                d_loss = d_loss_wgan + self.lambda_gp * gp

            # 판별자 그래디언트 계산 및 업데이트
            disc_gradients = disc_tape.gradient(d_loss, self.discriminator.trainable_variables)
            self.disc_optimizer.apply_gradients(zip(disc_gradients, self.discriminator.trainable_variables))

        # =======================================================
        # 2. 생성자(Generator) 학습
        # =======================================================
        with tf.GradientTape() as gen_tape:
            noise = tf.random.normal(shape=(batch_size, LATENT_DIM))
            # 새로운 가짜 이미지 생성
            pred = self.generator([selected_fire, terrain, selected_tnh, selected_wind, step_indices, noise], training=True)
            # 해당 이미지에 대한 판별자 점수
            disc_fake_score_for_gen = self.discriminator([pred, terrain, selected_tnh, selected_wind, step_indices], training=True)

            # 생성자 손실 (Adversarial)
            g_loss_adv = -tf.reduce_mean(disc_fake_score_for_gen)

            # L1 손실
            thr = 0.05
            fire_mask = tf.cast(selected_target > thr, pred.dtype)
            # 배경에도 약간의 가중치(0.3) 줘서 홀로우 현상 방지
            w = 0.3 + 0.7 * fire_mask
            l1_loss = tf.reduce_sum(w * tf.abs(selected_target - pred)) / (tf.reduce_sum(w) + 1e-6)
            #l1_loss = tf.reduce_mean(tf.abs(selected_target - pred))
            #tf.print("Pred min/max/mean:", tf.reduce_min(pred), tf.reduce_max(pred), tf.reduce_mean(pred))
            #tf.print("Target min/max/mean:", tf.reduce_min(selected_target), tf.reduce_max(selected_target), tf.reduce_mean(selected_target))

            # # Dice 손실
            axes = [1, 2, 3]
            intersection = tf.reduce_sum(selected_target * pred, axis=axes)
            union = tf.reduce_sum(selected_target, axis=axes) + tf.reduce_sum(pred, axis=axes)
            
            
            dice_loss = 1.0 - tf.reduce_mean((2.0 * intersection + 1e-6) / (union + 1e-6))
            # ★★★★★ 강제 테스트 ★★★★★
             
            # 최종 생성자 손실
            g_loss = g_loss_adv + self.lambda_L1 * l1_loss + self.lambda_dice * dice_loss 
        
        # 생성자 그래디언트 계산 및 업데이트
        gen_gradients = gen_tape.gradient(g_loss, self.generator.trainable_variables)
        self.gen_optimizer.apply_gradients(zip(gen_gradients, self.generator.trainable_variables))

        return {
            "g_loss": g_loss/2, 
            "d_loss": d_loss/2, 
            "adv_loss": g_loss_adv/2, 
            "l1_loss": l1_loss/2, 
            "dice_loss": dice_loss/2,
            "gradient_penalty": gp/2
        }

# ======================================
# 모델 생성 및 컴파일
# ======================================

with strategy.scope():
    gen = build_generator()
    disc = build_discriminator()
    progressive_gan = ProgressiveStepGAN(gen, disc)
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.5)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
    progressive_gan.compile(gen_optimizer=gen_optimizer, disc_optimizer=disc_optimizer)


print(gen.summary())
print(disc.summary())



In [ ]:
def _parse_function(example_proto):
    features = {
        'spread_0h': tf.io.FixedLenFeature([], tf.string),
        'spread_1h': tf.io.FixedLenFeature([], tf.string),
        'spread_2h': tf.io.FixedLenFeature([], tf.string),
        'spread_3h': tf.io.FixedLenFeature([], tf.string),
        'spread_4h': tf.io.FixedLenFeature([], tf.string),
        'spread_5h': tf.io.FixedLenFeature([], tf.string),
        'spread_6h': tf.io.FixedLenFeature([], tf.string),
        'spread_7h': tf.io.FixedLenFeature([], tf.string),
        'spread_8h': tf.io.FixedLenFeature([], tf.string),
        'spread_9h': tf.io.FixedLenFeature([], tf.string),
        'spread_10h': tf.io.FixedLenFeature([], tf.string),
        'spread_11h': tf.io.FixedLenFeature([], tf.string),
        'spread_12h': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'tnh': tf.io.FixedLenFeature([], tf.string),
        'wind': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    
    def decode_spread(key):
        img = tf.io.decode_raw(parsed[key], tf.float32)
        img = tf.reshape(img, [IMG_H, IMG_W, CHANNELS_FIRE])
        return img
    
    # 우리가 사용할 시간 포인트: 0h, 4h, 8h, 12h
    spread_0h  = decode_spread('spread_0h')
    spread_4h  = decode_spread('spread_4h')
    #spread_4h[spread_4h != 0] = 1
    spread_8h  = decode_spread('spread_8h')
    #spread_8h[spread_8h != 0] = 1
    spread_12h = decode_spread('spread_12h')
    #spread_12h[spread_12h != 0] = 1

    # fire_seq_sub: (4, IMG_H, IMG_W, 1) → 0h는 input, 4h,8h,12h는 타깃 예측용
    fire_seq_sub = tf.stack([spread_0h, spread_4h, spread_8h, spread_12h], axis=0)
    
    # 타깃: target_final은 spread_12h, target_seq는 [spread_4h, spread_8h, spread_12h] → (3, IMG_H, IMG_W, 1)
    target_final = spread_12h
    target_seq = tf.stack([spread_4h, spread_8h, spread_12h], axis=0)
    
    # terrain 이미지: (IMG_H, IMG_W, 3)
    terrain = tf.io.decode_raw(parsed['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [IMG_H, IMG_W, CHANNELS_TERRAIN])
    
    # dem, fuel (생략)
    dem = tf.io.decode_raw(parsed['dem_image'], tf.float32)
    dem = tf.reshape(dem, [IMG_H, IMG_W, 1])
    fuel = tf.io.decode_raw(parsed['fuel_image'], tf.float32)
    fuel = tf.reshape(fuel, [IMG_H, IMG_W, 1])
    
    # 기상 데이터: tnh와 wind → (12,4)
    tnh = tf.io.decode_raw(parsed['tnh'], tf.float32)
    tnh = tf.reshape(tnh, [12, 2])
    wind = tf.io.decode_raw(parsed['wind'], tf.float32)
    wind = tf.reshape(wind, [12, 2])
    weather_full = tf.concat([tnh, wind], axis=-1)  # (12,4)
    # 대신 평균 내지 않고, reshape: 12→ 3×4, so weather_seq becomes (3,4,4)
    tnh_seq = tf.reshape(tnh, [3, 4, 2])
    wind_seq = tf.reshape(wind, [3, 4, 2])
    
    ignite_index = parsed['ignite_index']
    weather_index = parsed['weather_index']
    
    # 반환: inputs와 targets
    # inputs: terrain, fire_seq_sub, weather_seq, ignite_index, weather_index
    # targets: target_final, target_seq
    return ((terrain+1)/2, (fire_seq_sub+1)/2, tnh_seq,wind_seq)

def _parse_function2(example_proto):
    features = {
        'spread_0h': tf.io.FixedLenFeature([], tf.string),
        'spread_1h': tf.io.FixedLenFeature([], tf.string),
        'spread_2h': tf.io.FixedLenFeature([], tf.string),
        'spread_3h': tf.io.FixedLenFeature([], tf.string),
        'spread_4h': tf.io.FixedLenFeature([], tf.string),
        'spread_5h': tf.io.FixedLenFeature([], tf.string),
        'spread_6h': tf.io.FixedLenFeature([], tf.string),
        'spread_7h': tf.io.FixedLenFeature([], tf.string),
        'spread_8h': tf.io.FixedLenFeature([], tf.string),
        'spread_9h': tf.io.FixedLenFeature([], tf.string),
        'spread_10h': tf.io.FixedLenFeature([], tf.string),
        'spread_11h': tf.io.FixedLenFeature([], tf.string),
        'spread_12h': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'tnh': tf.io.FixedLenFeature([], tf.string),
        'wind': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    
    def decode_spread(key):
        img = tf.io.decode_raw(parsed[key], tf.float32)
        img = tf.reshape(img, [IMG_H, IMG_W, CHANNELS_FIRE])
        return img
    
    # 우리가 사용할 시간 포인트: 0h, 4h, 8h, 12h
    spread_0h  = decode_spread('spread_0h')
    spread_4h  = decode_spread('spread_4h')
    #spread_4h[spread_4h != 0] = 1
    spread_8h  = decode_spread('spread_8h')
    #spread_8h[spread_8h != 0] = 1
    spread_12h = decode_spread('spread_12h')
    #spread_12h[spread_12h != 0] = 1

    # fire_seq_sub: (4, IMG_H, IMG_W, 1) → 0h는 input, 4h,8h,12h는 타깃 예측용
    fire_seq_sub = tf.stack([spread_0h, spread_4h, spread_8h, spread_12h], axis=0)
    
    # 타깃: target_final은 spread_12h, target_seq는 [spread_4h, spread_8h, spread_12h] → (3, IMG_H, IMG_W, 1)
    target_final = spread_12h
    target_seq = tf.stack([spread_4h, spread_8h, spread_12h], axis=0)
    
    # terrain 이미지: (IMG_H, IMG_W, 3)
    terrain = tf.io.decode_raw(parsed['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [IMG_H, IMG_W, CHANNELS_TERRAIN])
    
    # dem, fuel (생략)
    dem = tf.io.decode_raw(parsed['dem_image'], tf.float32)
    dem = tf.reshape(dem, [IMG_H, IMG_W, 1])
    fuel = tf.io.decode_raw(parsed['fuel_image'], tf.float32)
    fuel = tf.reshape(fuel, [IMG_H, IMG_W, 1])
    
    # 기상 데이터: tnh와 wind → (12,4)
    tnh = tf.io.decode_raw(parsed['tnh'], tf.float32)
    tnh = tf.reshape(tnh, [12, 2])
    wind = tf.io.decode_raw(parsed['wind'], tf.float32)
    wind = tf.reshape(wind, [12, 2])
    weather_full = tf.concat([tnh, wind], axis=-1)  # (12,4)
    # 대신 평균 내지 않고, reshape: 12→ 3×4, so weather_seq becomes (3,4,4)
    tnh_seq = tf.reshape(tnh, [3, 4, 2])
    wind_seq = tf.reshape(wind, [3, 4, 2])
    
    ignite_index = parsed['ignite_index']
    weather_index = parsed['weather_index']
    
    # 반환: inputs와 targets
    # inputs: terrain, fire_seq_sub, weather_seq, ignite_index, weather_index
    # targets: target_final, target_seq
    return (terrain, fire_seq_sub, tnh_seq, wind_seq)


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt



# tfrecord 파일 목록 (예시)
#tfrecord_files = ["./ignition/test_128_56ea.tfrecord"]
tfrecord_files = ["./train_big_128_all_origin_mix.tfrecord"]
# 데이터셋 생성 및 매핑
train_dataset = tf.data.TFRecordDataset(tfrecord_files)
train_dataset = train_dataset.shuffle(buffer_size=4096)
train_dataset = train_dataset.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(64, drop_remainder=True)  # 시각화를 위해 1개 배치로 설정
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
#train_dataset = train_dataset.take(3000)

# 하나의 배치를 가져와서 시각화
for (inputs) in train_dataset.take(1):
    (terrain, fire_seq_sub, tnh_seq,wind_seq) = inputs
    print("Terrain shape:", terrain.shape)
    print("Fire seq shape:", fire_seq_sub.shape)
    print("Weather seq shape:", tnh_seq.shape)
    print("Weather seq shape:", wind_seq.shape)
    plt.figure(figsize=(20,5))
    plt.subplot(1,4,1)
    plt.imshow(terrain[0])
    print(tf.reduce_max(terrain[0]))
    print(tf.reduce_min(terrain[0]))
    plt.title("Terrain")
    plt.axis("off")
    plt.subplot(1,4,2)
    plt.imshow(fire_seq_sub[0,1].numpy().squeeze(), cmap='gray')
    plt.title("Fire 0h")
    plt.axis("off")
    plt.subplot(1,4,3)
    plt.imshow(fire_seq_sub[0,2].numpy().squeeze(), cmap='gray')
    plt.title("Target (4h)")
    plt.axis("off")
    plt.subplot(1,4,4)
    plt.imshow(fire_seq_sub[0,3].numpy().squeeze() , cmap='gray')
    print(tf.reduce_max(fire_seq_sub[0,3]))
    print(tf.reduce_min(fire_seq_sub[0,3]))
    plt.title("Target (8h)")
    plt.axis("off")
    plt.show()
    break


In [ ]:
class SaveStepwisePredictions(tf.keras.callbacks.Callback):
    def __init__(self, sample_dataset, output_dir='./predictions_mix', freq=1, n_ensemble=5):
        super().__init__()
        
        # ★★★ 1. 배치 데이터에서 첫 번째 샘플만 저장 (배치 크기 문제 해결) ★★★
        sample_data = next(iter(sample_dataset.take(1)))
        (terrain_batch, fire_seq_batch, tnh_seq_batch, wind_seq_batch) = sample_data
        
        self.terrain     = terrain_batch[0:1]    # (1, H, W, 3)
        self.fire_seq    = fire_seq_batch[0:1]   # (1, 4, H, W, 1)
        self.tnh_seq     = tnh_seq_batch[0:1]    # (1, 3, 4, 2)
        self.wind_seq    = wind_seq_batch[0:1]   # (1, 3, 4, 2)
        
        self.output_dir  = output_dir
        self.freq        = freq
        self.n_ensemble  = n_ensemble # 앙상블 횟수
        os.makedirs(output_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.freq != 0:
            return

        model_to_call = getattr(self.model, 'generator', self.model)

        # 초기 산불 상태
        current_fire_input = self.fire_seq[:, 0]  # (1, H, W, 1)

        # 예측 결과를 저장할 리스트
        mean_preds = []
        variance_preds = []
        
        # 3단계(4h, 8h, 12h) 순서대로 예측
        for step in range(3):
            step_onehot = tf.one_hot([step], depth=3)
            selected_tnh  = self.tnh_seq[:, step]
            selected_wind = self.wind_seq[:, step]
            
            # ★★★ 2. N번의 앙상블 예측 수행 ★★★
            step_ensemble_preds = []
            for _ in range(self.n_ensemble):
                # 매번 새로운 노이즈 생성
                noise = tf.random.normal(shape=(1, LATENT_DIM))
                
                # 예측
                pred = model_to_call(
                    [current_fire_input, self.terrain, selected_tnh, selected_wind, step_onehot, noise],
                    training=False
                ) # (1, H, W, 1)
                step_ensemble_preds.append(pred[0]) # (H, W, 1)

            # 앙상블 결과 종합
            ensemble_tensor = tf.stack(step_ensemble_preds, axis=0) # (N, H, W, 1)

            # ★★★ 3. 평균(확률)과 분산(불확실성) 계산 ★★★
            mean_pred = tf.reduce_mean(ensemble_tensor, axis=0)         # (H, W, 1)
            variance_pred = tf.math.reduce_variance(ensemble_tensor, axis=0) # (H, W, 1)

            mean_preds.append(mean_pred[..., 0])
            variance_preds.append(variance_pred[..., 0])

            # ★★★ 4. 다음 단계 입력으로 '평균' 예측 결과 사용 ★★★
            # 모델 출력이 '도달 시간'이므로, 특정 시간(4h)까지 도달했는지 여부로 변환
            # (출력이 0~1 확률이라면, 0.5 임계값 적용)
            time_threshold = 4.0 # 현재 단계의 시간 (예: 4h, 8h, 12h)
            
            # 이 부분은 모델의 정확한 출력(도달 시간, 확률 등)에 따라 맞게 수정해야 합니다.
            # 여기서는 출력이 0~1 사이의 확률이라고 가정합니다.
            binarized_pred = tf.where(mean_pred > 0.5, 1.0, 0.0)
            current_fire_input = tf.expand_dims(binarized_pred, axis=0)

        # 실제 정답 마스크
        truths = [self.fire_seq[0, i, ..., 0] for i in range(1, 4)]
        
        # ★★★ 5. 시각화 코드 수정 ★★★
        hours = [4, 8, 12]
        fig, axes = plt.subplots(3, 3, figsize=(9, 9)) # 3행 3열로 변경
        
        for i, hr in enumerate(hours):
            # 1열: 실제 정답
            axes[i, 0].imshow(truths[i], cmap='hot', vmin=0, vmax=1)
            axes[i, 0].set_title(f'True {hr}h')
            axes[i, 0].axis('off')
            
            # 2열: 평균 예측 (확률 맵)
            im_mean = axes[i, 1].imshow(mean_preds[i], cmap='hot', vmin=0, vmax=1)
            axes[i, 1].set_title(f'Pred Mean {hr}h')
            axes[i, 1].axis('off')
            
            # 3열: 분산 예측 (불확실성 맵)
            im_var = axes[i, 2].imshow(variance_preds[i], cmap='magma') # 다른 컬러맵 사용
            axes[i, 2].set_title(f'Pred Variance {hr}h')
            axes[i, 2].axis('off')

        plt.tight_layout()
        outpath = os.path.join(self.output_dir, f'epoch_{epoch+1:03d}.png')
        fig.savefig(outpath, bbox_inches='tight')
        plt.close(fig)

# --- 사용법 예시 ---
# 1) 고정 샘플 준비 (데이터셋에서 1개 배치 가져오기)
sample_dataset = train_dataset.take(1)

# 2) 콜백 인스턴스 생성
save_cb = SaveStepwisePredictions(sample_dataset, freq=1, n_ensemble=6)
(terrain, fire_seq_sub, tnh_seq,wind_seq) = next(iter(sample_dataset.take(1)))
print("Terrain shape:", terrain.shape)
print("Fire seq shape:", fire_seq_sub.shape)
print("Weather seq shape:", tnh_seq.shape)
print("Weather seq shape:", wind_seq.shape)
plt.figure(figsize=(20,5))
plt.subplot(1,4,1)
plt.imshow(terrain[0])
plt.title("Terrain")
plt.axis("off")
plt.subplot(1,4,2)
plt.imshow(fire_seq_sub[0,1].numpy().squeeze(), cmap='gray')
plt.title("Fire 0h")
plt.axis("off")
plt.subplot(1,4,3)
plt.imshow(fire_seq_sub[0,2].numpy().squeeze(), cmap='gray')
plt.title("Target (4h)")
plt.axis("off")
plt.subplot(1,4,4)
plt.imshow(fire_seq_sub[0,3].numpy().squeeze() , cmap='gray')
plt.title("Target (8h)")
plt.axis("off")
plt.show()

In [ ]:
import os, datetime
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, CSVLogger

run_dir = os.path.join("runs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(run_dir, exist_ok=True)

tb_cb  = TensorBoard(log_dir=run_dir, histogram_freq=0, write_graph=False, write_images=False)

In [ ]:

progressive_gan.fit(train_dataset, epochs=50,callbacks=[save_cb])

In [ ]:
gen.save('./lstm_v_G_mix.keras')
disc.save('./lstm_v_D_mix.keras')
gen.save('./lstm_v_G_mix.h5')
disc.save('./lstm_v_D_mix.h5')

In [ ]:
generator = tf.keras.models.load_model('./ignition/lstm_v_G.keras')
discriminator = tf.keras.models.load_model('./ignition/lstm_v_D.keras')

In [ ]:
generator.save('root/gene')

In [ ]:
gen.load_weights('./lstm_v_G_mix.h5')
disc.load_weights('./lstm_v_D_mix.h5')

In [ ]:
gen = tf.keras.models.load_model('./ignition/lstm_v_G.keras')

In [ ]:
gen.summary()

In [ ]:
import tensorflow as tf
import os

# 원본 TFRecord 파일 경로
source_tfrecord = "./ignition/test_small_128_all.tfrecord"
# 저장할 샘플 파일 경로
sample_tfrecord_path = "./sample_data.tfrecord"

# 파싱하지 않은 원본 데이터셋을 로드합니다.
raw_dataset = tf.data.TFRecordDataset([source_tfrecord])

# 데이터셋에서 첫 번째 raw example 하나만 가져옵니다.
sample_raw_data = next(iter(raw_dataset))

# 가져온 raw example을 새로운 TFRecord 파일에 씁니다.
with tf.io.TFRecordWriter(sample_tfrecord_path) as writer:
    # .numpy()를 사용해 EagerTensor에서 byte string을 추출합니다.
    writer.write(sample_raw_data.numpy())

print(f"샘플 데이터가 '{sample_tfrecord_path}'에 성공적으로 저장되었습니다. ✅")

In [ ]:
def ensemble_chaining(generator, terrain, current_fire, tnh_seq, wind_seq, n_ensemble=6):
    
    mean_predictions = []
    current_fire_input = current_fire # (batch, H, W, C)
    
    # 4h, 8h, 12h 예측 (총 3 스텝)
    # tnh_seq.shape[1]은 4 (0h, 4h, 8h, 12h) 이므로 range(4)가 될 것.
    # 하지만 예측은 4h, 8h, 12h (3스텝)이므로 t+1을 사용하기 위해 range(3)으로 제한합니다.
    for t in range(3): # 0, 1, 2 에 해당 (각각 4h, 8h, 12h 예측)
        step_indices = tf.one_hot(t, depth=3) # t는 0, 1, 2
        step_indices = tf.expand_dims(step_indices, axis=0) # (1, 3)

        # tnh_seq와 wind_seq에서 해당 시간 스텝 데이터 추출
        # tnh_seq: (batch, 4, 1, 2) -> tnh_seq[:, t+1, ...] 는 (batch, 1, 2)
        # wind_seq: (batch, 4, 2, 1) -> wind_seq[:, t+1, ...] 는 (batch, 2, 1)
        # 예측은 fire_seq의 t+1 스텝 (4h, 8h, 12h)의 날씨 정보를 사용해야 함.
        tnh_t = tnh_seq[:, t]  # 0h 인덱스가 아니라 예측하려는 스텝의 날씨 정보
        wind_t = wind_seq[:, t]
        
        # --- 앙상블 예측 ---
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM)) # 배치 크기 1에 맞춰 (1, LATENT_DIM)
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0) # (n_ensemble, H, W, C)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0) # (H, W, C)
        mean_predictions.append(tf.expand_dims(mean_pred, axis=0)) # (1, H, W, C) -> 각 스텝 예측 저장
        
        # 다음 스텝의 입력으로 현재 예측 결과를 사용
        # 현재 mean_pred는 (H, W, C) 이므로, current_fire_input (1, H, W, C)로 만들어야 함
        current_fire_input = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C)

    # 결과를 쌓아서 반환
    # mean_predictions: 각 요소가 (1, H, W, C) -> stacked_means: (3, 1, H, W, C) -> tf.stack(axis=1)하면 (1, 3, H, W, C)
    stacked_means = tf.stack(mean_predictions, axis=1) # (batch, num_steps, H, W, C)
    

    return stacked_means

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from scipy.linalg import sqrtm
from tqdm import tqdm

def get_mask(y_true, y_pred, threshold=0.1, mode="union"):
    """
    불난 영역 마스크를 만든다.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    if mode == "gt":
        mask = y_true_bin
    elif mode == "pred":
        mask = y_pred_bin
    elif mode == "inter":
        mask = y_true_bin * y_pred_bin
    else:  # "union"
        mask = tf.cast((y_true_bin + y_pred_bin) > 0.0, tf.float32)

    return mask  # same shape as y_true, {0,1}


def masked_mse(y_true, y_pred, mask):
    """
    마스크 영역 내에서만 MSE 계산.
    (픽셀별 '도달 시간' 차이를 그대로 살림)
    """
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)

    mse_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_sum(diff2) / denom
    )
    return mse_val


def masked_ssim(y_true, y_pred, mask, max_val=1.0):
    """
    마스크 밖은 (0,0)으로 맞춰서 SSIM이 거기선 영향 없게 하고,
    마스크 안 영역의 구조(형태/경계)를 비교.
    """
    y_true_focus = y_true * mask
    y_pred_focus = y_pred * mask

    ssim_val = tf.image.ssim(y_true_focus, y_pred_focus, max_val=max_val)

    denom = tf.reduce_sum(mask)
    ssim_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_mean(ssim_val)
    )
    return ssim_val


def calculate_iou(y_true, y_pred, threshold=0.1):
    """
    번진 영역(불 도달) 자체의 IoU.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou

# --- FID 계산을 위한 함수 (새로 추가) ---
def calculate_fid(model, real_images, generated_images):
    """
    실제 이미지와 생성 이미지 세트 간의 FID 점수를 계산합니다.
    이미지는 InceptionV3에 맞게 전처리됩니다.
    """
    # InceptionV3 모델 입력 형식에 맞게 이미지 전처리
    # 1. 3채널로 복제 (원본이 1채널이므로)
    # 2. 크기 조정 (최소 75x75 필요)
    # 3. Keras의 전처리 함수 적용
    def preprocess_images(images):
        # images: (N, H, W, 1)
        images_resized = tf.image.resize(images, [75, 75]) # InceptionV3 최소 크기
        images_rgb = tf.image.grayscale_to_rgb(images_resized) # 3채널로 복제
        images_preprocessed = preprocess_input(images_rgb)
        return images_preprocessed

    real_processed = preprocess_images(real_images)
    gen_processed = preprocess_images(generated_images)

    # InceptionV3 모델을 통해 특징 추출
    real_features = model.predict(real_processed)
    gen_features = model.predict(gen_processed)

    # 각 특징 벡터의 평균과 공분산 계산
    mu_real, sigma_real = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu_gen, sigma_gen = np.mean(gen_features, axis=0), np.cov(gen_features, rowvar=False)

    # FID 점수 계산
    ssdiff = np.sum((mu_real - mu_gen)**2.0)
    covmean = sqrtm(sigma_real.dot(sigma_gen))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma_real + sigma_gen - 2.0 * covmean)
    return fid

# --- IoU 계산을 위한 함수 (새로 추가) ---
def calculate_iou(y_true, y_pred, threshold=0.1):
    """
    임계값을 기준으로 이진화한 후 IoU를 계산합니다.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7) # 0으로 나누는 것을 방지
    return iou.numpy().item()



def ensemble_chaining_with_disc_scores(generator, discriminator, 
                                     terrain, fire_seq, tnh_seq, wind_seq, 
                                     n_ensemble=5):
    """
    앙상블 체이닝 추론을 수행하며, 각 단계별 판별자 점수를 함께 반환합니다. (Shape 오류 수정 버전)
    """
    # fire_seq: (batch, time_steps, H, W, C) -> fire_seq[:, 0, ...]는 (batch, H, W, C)
    initial_fire_input = fire_seq[:, 0, ...] # 0h (현재) fire state
    
    mean_predictions = []
    variance_predictions = []
    real_scores = []
    fake_scores = []
    
    current_fire_input = initial_fire_input # (batch, H, W, C)
    
    # 4h, 8h, 12h 예측 (총 3 스텝)
    # tnh_seq.shape[1]은 4 (0h, 4h, 8h, 12h) 이므로 range(4)가 될 것.
    # 하지만 예측은 4h, 8h, 12h (3스텝)이므로 t+1을 사용하기 위해 range(3)으로 제한합니다.
    for t in range(3): # 0, 1, 2 에 해당 (각각 4h, 8h, 12h 예측)
        step_indices = tf.one_hot(t, depth=3) # t는 0, 1, 2
        step_indices = tf.expand_dims(step_indices, axis=0) # (1, 3)

        # tnh_seq와 wind_seq에서 해당 시간 스텝 데이터 추출
        # tnh_seq: (batch, 4, 1, 2) -> tnh_seq[:, t+1, ...] 는 (batch, 1, 2)
        # wind_seq: (batch, 4, 2, 1) -> wind_seq[:, t+1, ...] 는 (batch, 2, 1)
        # 예측은 fire_seq의 t+1 스텝 (4h, 8h, 12h)의 날씨 정보를 사용해야 함.
        tnh_t = tnh_seq[:, t]  # 0h 인덱스가 아니라 예측하려는 스텝의 날씨 정보
        wind_t = wind_seq[:, t]
        
        # --- 앙상블 예측 ---
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM)) # 배치 크기 1에 맞춰 (1, LATENT_DIM)
            
            # 제너레이터 입력: [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise]
            # current_fire_input: (1, H, W, C)
            # terrain: (1, H, W, C)
            # tnh_t: (1, 1, 2)
            # wind_t: (1, 2, 1)
            # step_indices: (1, 3)
            # noise: (1, LATENT_DIM)
            
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0) # (n_ensemble, H, W, C)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0) # (H, W, C)
        variance_pred = tf.math.reduce_variance(ensemble_tensor, axis=0) # (H, W, C)

        mean_predictions.append(tf.expand_dims(mean_pred, axis=0)) # (1, H, W, C) -> 각 스텝 예측 저장
        variance_predictions.append(tf.expand_dims(variance_pred, axis=0)) # (1, H, W, C)

        # --- 판별자 점수 계산 ---
        mean_pred_batched = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C) - 판별자 입력을 위해 배치 차원 추가
        
        # true_next_fire: fire_seq에서 예측하려는 다음 실제 불 이미지 (fire_seq[:, t+1, ...])
        true_next_fire = fire_seq[:, t + 1, ...] # (1, H, W, C)
        
        disc_inputs_real = [true_next_fire, terrain, tnh_t, wind_t, step_indices]
        disc_inputs_fake = [mean_pred_batched, terrain, tnh_t, wind_t, step_indices]

        # discriminator는 여러 개의 패치에 대한 점수를 반환할 수 있으므로, 평균을 취함
        real_score_patches = discriminator(disc_inputs_real, training=False) # (1, num_patches) 또는 (1, 1)
        fake_score_patches = discriminator(disc_inputs_fake, training=False) # (1, num_patches) 또는 (1, 1)
        
        avg_real_score = tf.reduce_mean(real_score_patches) # 스칼라
        avg_fake_score = tf.reduce_mean(fake_score_patches) # 스칼라
        
        real_scores.append(avg_real_score)
        fake_scores.append(avg_fake_score)
        
        # 다음 스텝의 입력으로 현재 예측 결과를 사용
        # 현재 mean_pred는 (H, W, C) 이므로, current_fire_input (1, H, W, C)로 만들어야 함
        current_fire_input = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C)

    # 결과를 쌓아서 반환
    # mean_predictions: 각 요소가 (1, H, W, C) -> stacked_means: (3, 1, H, W, C) -> tf.stack(axis=1)하면 (1, 3, H, W, C)
    stacked_means = tf.stack(mean_predictions, axis=1) # (batch, num_steps, H, W, C)
    stacked_vars = tf.stack(variance_predictions, axis=1) # (batch, num_steps, H, W, C)
    
    # real_scores/fake_scores는 각 스텝별 스칼라 텐서 리스트 (3개)
    return stacked_means, stacked_vars, real_scores, fake_scores

# --- 메인 코드 (수정 불필요) ---

# 테스트 데이터셋 불러오기
test_tfrecord = "./test_big_128_all_origin_mix.tfrecord"
#test_tfrecord = "./ignition/test_last_128_53ea.tfrecord"
test_dataset = tf.data.TFRecordDataset([test_tfrecord])
test_dataset = test_dataset.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
test_list = list(test_dataset)
num_batches = len(test_list)
print(len(test_list))
# 원하는 샘플 개수 선택
# --- 원하는 샘플 개수와 시작 인덱스 설정 ---
NN = 0 # 0부터 시작, 0이면 0~35, 1이면 36~71 등 36개 단위로 이동
start_sample_idx = 28 * NN # 0부터 시작하여 시각화를 시작할 샘플의 인덱스
num_samples_to_visualize = 28 # 원하는 샘플 수 (예: 1~36)

# 인덱스 유효성 검사 및 조정
if start_sample_idx >= num_batches:
    print(f"시작 인덱스 {start_sample_idx}가 전체 샘플 수 {num_batches}보다 크거나 같습니다. 시각화할 샘플이 없습니다.")
    exit()

if start_sample_idx + num_samples_to_visualize > num_batches:
    num_samples_to_visualize = num_batches - start_sample_idx
    print(f"경고: 요청된 샘플 수가 전체 데이터셋을 초과합니다. {num_samples_to_visualize}개의 샘플만 시각화합니다.")

if num_samples_to_visualize <= 0:
    print("시각화할 샘플이 없습니다. 시작 인덱스와 샘플 수를 확인하세요.")
    exit()

# 순차적인 인덱스 리스트 생성
sequential_batch_idxs = list(range(start_sample_idx, start_sample_idx + num_samples_to_visualize))

inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(75, 75, 3))
all_gt_images_cgan = []
all_pred_images_cgan = []

# --- 시각화 설정 ---
# 각 샘플은 2행 (실제, 예측) x 3열 (4h, 8h, 12h)을 차지합니다.
num_cols_per_sample_row = 3 # 4h, 8h, 12h
num_rows_per_sample = 2 # 실제 이미지, 예측 평균 이미지

total_plot_cols = num_cols_per_sample_row # 전체 subplot 그리드의 열 수 (3)
total_plot_rows = num_samples_to_visualize * num_rows_per_sample # 전체 subplot 그리드의 행 수 (샘플 수 * 2)

# Figure 크기 조정
# 각 서브플롯당 대략적인 크기를 5x5 인치로 가정하여 전체 Figure 크기를 계산합니다.
plt.figure(figsize=(total_plot_cols * 5, total_plot_rows * 5)) 

print(f"\n총 {num_samples_to_visualize}개의 샘플을 시각화합니다 (인덱스 {start_sample_idx}부터).")
print(f"생성될 시각화 그리드: {total_plot_rows} 행 x {total_plot_cols} 열")

for i, current_sample_relative_idx in enumerate(range(num_samples_to_visualize)):
    bidx = sequential_batch_idxs[current_sample_relative_idx]
    terrain, fire_seq, tnh_seq, wind_seq = test_list[bidx]

    print(f"\n--- Processing Sample Index: {bidx} ---")

    start_time = time.time()
    mean_preds, variance_preds, real_scores, fake_scores = ensemble_chaining_with_disc_scores(
        gen, disc, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=4
    )
    end_time = time.time()
    prediction_time = end_time - start_time
    print(f"Prediction Time for Sample {bidx}: {prediction_time:.2f} seconds")

    mean_preds_np = mean_preds.numpy()
    target_seq_np = fire_seq[0, 1:4].numpy()
    real_scores_np = np.array([score.numpy() for score in real_scores])
    fake_scores_np = np.array([score.numpy() for score in fake_scores])

    # FID 계산을 위해 현재 배치의 이미지들 저장
    all_gt_images_cgan.extend(target_seq_np) # (3, H, W, C) 이미지가 추가됨
    all_pred_images_cgan.extend(mean_preds_np[0]) # (3, H, W, C) 이미지가 추가됨

    num_steps = mean_preds_np.shape[1]
    hours = [4, 8, 12]

    current_sample_start_row = i * 2

    for t in range(num_steps):
        gt_img_tensor = tf.convert_to_tensor(target_seq_np[t], dtype=tf.float32)
        mean_img_tensor = tf.convert_to_tensor(mean_preds_np[0, t], dtype=tf.float32)

        gt_b  = tf.expand_dims(gt_img_tensor, axis=0)
        pred_b = tf.expand_dims(mean_img_tensor, axis=0)

        # 마스크 생성 (union / threshold=0.1 그대로)
        mask = get_mask(gt_b, pred_b, threshold=0.1, mode="union")

        # MSE (마스크 안에서만)
        mse_val = masked_mse(gt_b, pred_b, mask)
        mse_score = mse_val.numpy().item()  # float

        # SSIM (마스크 밖 영향 제거)
        ssim_val = masked_ssim(gt_b, pred_b, mask, max_val=1.0)
        ssim_score = ssim_val.numpy().item()  # float

        # IoU (불 번짐 영역의 이진 IoU)
        iou_val = calculate_iou(gt_b, pred_b, threshold=0.1)
        iou_score = iou_val

        gt_img = np.squeeze(gt_img_tensor.numpy())
        mean_img = np.squeeze(mean_img_tensor.numpy())

        # 실제 이미지 subplot
        plt.subplot(total_plot_rows, total_plot_cols, current_sample_start_row * total_plot_cols + t + 1)
        plt.imshow(gt_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"True {hours[t]}h (Idx {bidx})\n(Score: {real_scores_np[t]:.3f})")
        plt.axis('off')

        # 예측 이미지 subplot (제목에 IoU 추가)
        plt.subplot(total_plot_rows, total_plot_cols, (current_sample_start_row + 1) * total_plot_cols + t + 1)
        plt.imshow(mean_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"Pred Mean {hours[t]}h (Idx {bidx})\n"
                  f"(D-Score: {fake_scores_np[t]:.3f}, MSE: {mse_score:.4f}\n"
                  f"SSIM: {ssim_score:.4f}, IoU: {iou_score:.4f})") # IoU 추가
        plt.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# --- FID 점수 계산 및 출력 (루프 종료 후) ---
print("\n--- Calculating FID Score for CGAN Model ---")
# 리스트를 텐서로 변환 (N, H, W, C)
all_gt_tensor_cgan = tf.convert_to_tensor(all_gt_images_cgan, dtype=tf.float32)
all_pred_tensor_cgan = tf.convert_to_tensor(all_pred_images_cgan, dtype=tf.float32)

# FID 계산
fid_score_cgan = calculate_fid(inception_model, all_gt_tensor_cgan, all_pred_tensor_cgan)
print(f"FID Score (CGAN vs. Ground Truth): {fid_score_cgan:.4f}")

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# =========================================================
# 0. 유틸 함수들 (기존 + 추가)
# =========================================================

def get_mask(y_true, y_pred, threshold=0.1, mode="union"):
    """
    불난 영역 마스크를 만든다.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    if mode == "gt":
        mask = y_true_bin
    elif mode == "pred":
        mask = y_pred_bin
    elif mode == "inter":
        mask = y_true_bin * y_pred_bin
    else:  # "union"
        mask = tf.cast((y_true_bin + y_pred_bin) > 0.0, tf.float32)

    return mask  # same shape as y_true, {0,1}


def masked_mse(y_true, y_pred, mask):
    """
    마스크 영역 내에서만 MSE 계산.
    (픽셀별 '도달 시간' 차이를 그대로 살림)
    """
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)

    mse_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_sum(diff2) / denom
    )
    return mse_val


def masked_ssim(y_true, y_pred, mask, max_val=1.0):
    """
    마스크 밖은 (0,0)으로 맞춰서 SSIM이 거기선 영향 없게 하고,
    마스크 안 영역의 구조(형태/경계)를 비교.
    """
    y_true_focus = y_true * mask
    y_pred_focus = y_pred * mask

    ssim_val = tf.image.ssim(y_true_focus, y_pred_focus, max_val=max_val)

    denom = tf.reduce_sum(mask)
    ssim_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_mean(ssim_val)
    )
    return ssim_val


def calculate_iou(y_true, y_pred, threshold=0.1):
    """
    번진 영역(불 도달) 자체의 IoU.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou


# -----------------
# 화선(경계선) 관련 추가 지표
# -----------------

def _to_hw(x):
    """
    내부 계산 편하게 (B,H,W,1) 또는 (H,W,1) 또는 (H,W)를 (H,W) float32로 변환
    """
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if len(x.shape) == 4:  # (B,H,W,1)
        x = x[0, ..., 0]
    elif len(x.shape) == 3:  # (H,W,1)
        x = x[..., 0]
    # else (H,W) -> 그대로
    return x  # (H,W)


def get_fire_area_binary(arrival_map, threshold=0.1):
    """
    arrival_map: 픽셀값이 '도달 시간(0~1)' 또는 0(미도달)인 맵.
    threshold 이하라면 '불이 이미 도달했다'고 간주.
    여기서는 단순히 arrival_map > threshold 로 burn 여부를 판단하던 로직과 호환시키려면
    네 데이터 정의에 맞춰서 threshold 조건을 조정하면 된다.

    지금까지 코드는 (value > thr) == '탔다'를 가정했으니까 그대로 쓸게.
    즉 'thr_mask' 기준으로 "불 번짐 영역"을 정의한다.
    """
    area_bin = tf.cast(arrival_map > threshold, tf.float32)  # (H,W)
    return area_bin  # 0/1


def get_fireline_edge(area_bin):
    """
    area_bin: (H,W) float32 0/1
    반환: edge_bin (H,W) float32 0/1
    """
    # (H,W) -> (1,H,W,1)
    x = tf.expand_dims(area_bin, axis=0)    # (1,H,W)
    x = tf.expand_dims(x, axis=-1)          # (1,H,W,1)

    # dilation: max pooling
    dil = tf.nn.max_pool(
        x,
        ksize=[1, 3, 3, 1],
        strides=[1, 1, 1, 1],
        padding='SAME'
    )

    # erosion: min pooling via -max_pool(-x)
    ero = -tf.nn.max_pool(
        -x,
        ksize=[1, 3, 3, 1],
        strides=[1, 1, 1, 1],
        padding='SAME'
    )

    # morphological gradient 근사
    edge = tf.clip_by_value(dil - ero, 0.0, 1.0)  # (1,H,W,1)

    edge = tf.squeeze(edge, axis=0)   # (H,W,1)
    edge = tf.squeeze(edge, axis=-1)  # (H,W)

    edge_bin = tf.cast(edge > 0.5, tf.float32)    # 얇은 경계선만 1
    return edge_bin  # (H,W)



def fireline_length(edge_binary, pixel_size=1.0):
    """
    화선 길이 근사.
    pixel_size: 한 픽셀의 실제 길이(예: m). 아직 없으면 1.0으로 두고 상대 비교만 해도 됨.
    """
    return tf.reduce_sum(edge_binary) * pixel_size


def boundary_iou(edge_true, edge_pred):
    """
    경계선 IoU. (화선끼리 얼마나 겹치는지)
    """
    inter = tf.reduce_sum(edge_true * edge_pred)
    union = tf.reduce_sum(edge_true) + tf.reduce_sum(edge_pred) - inter
    biou = (inter + 1e-7) / (union + 1e-7)
    return biou


def chamfer_distance(edge_true, edge_pred):
    """
    화선 간 평균 거리 (대칭 Chamfer distance).
    값이 작을수록 예측 화선이 실제 화선 위치에 가깝다.
    단위: 픽셀 (pixel_size 곱하면 실제 거리로도 가능)
    """
    # 좌표 추출
    coords_true = tf.where(edge_true > 0.5)  # (N1,2) [y,x]
    coords_pred = tf.where(edge_pred > 0.5)  # (N2,2)

    # 화선이 아예 없으면 NaN 처리
    cond_empty = tf.logical_or(
        tf.equal(tf.shape(coords_true)[0], 0),
        tf.equal(tf.shape(coords_pred)[0], 0)
    )
    def _nan():
        return tf.constant(np.nan, dtype=tf.float32)

    def _compute():
        # 거리 행렬 (N1,N2)
        diff = tf.cast(tf.expand_dims(coords_true, 1) - tf.expand_dims(coords_pred, 0), tf.float32)
        dist = tf.sqrt(tf.reduce_sum(tf.square(diff), axis=2))  # (N1,N2)

        # true->pred 평균 최근접거리
        d1 = tf.reduce_mean(tf.reduce_min(dist, axis=1))
        # pred->true 평균 최근접거리
        d2 = tf.reduce_mean(tf.reduce_min(dist, axis=0))
        return (d1 + d2) / 2.0

    return tf.cond(cond_empty, _nan, _compute)


# =========================================================
# 1. 예측 함수 (기존)
# =========================================================
def get_ensemble_predictions(generator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=9):
    """
    하나의 샘플에 대해 자기회귀 및 앙상블 예측을 수행합니다.
    결과: (B,3,H,W,1)  -> [4h 예측, 8h 예측, 12h 예측]
    """
    initial_fire_input = fire_seq[:, 0, ...]  # (B,H,W,1)
    mean_predictions = []
    current_fire_input = initial_fire_input

    for t in range(3):  # predict 4h,8h,12h
        step_indices = tf.one_hot(t, depth=3)          # (3,)
        step_indices = tf.expand_dims(step_indices, 0) # (1,3)

        tnh_t = tnh_seq[:, t]   # (B,4,2) 등
        wind_t = wind_seq[:, t] # (B,4,2) 등

        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM))
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0)  # (n_ens,B,H,W,1) where B=1
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0)       # (B,H,W,1)
        mean_predictions.append(tf.expand_dims(mean_pred, axis=0))# (1,1,H,W,1)
        current_fire_input = tf.expand_dims(mean_pred, axis=0)

    return tf.stack(mean_predictions, axis=1)  # (B,3,H,W,1)


# =========================================================
# 2. 평가 함수 (업그레이드)
# =========================================================
def evaluate_model(generator, dataset, n_ensemble=9, thr_mask=0.1, pixel_size=1.0):
    """
    dataset 전체에 대해 시간 스텝별로:
    - masked MSE
    - masked SSIM
    - IoU
    - PerimeterRelativeError (화선 복잡도/둘레 길이)
    - BoundaryIoU (화선 위치 일치도)
    - ChamferDistance (화선 평균 거리 오차)

    반환 딕셔너리는 각 지표마다 [4h,8h,12h]에 대한 평균값 리스트를 가진다.
    """

    # 불 영역 기반 지표
    all_mse_scores   = [[] for _ in range(3)]
    all_ssim_scores  = [[] for _ in range(3)]
    all_iou_scores   = [[] for _ in range(3)]

    # 화선 기반 지표
    all_perim_relerr = [[] for _ in range(3)]
    all_biou_scores  = [[] for _ in range(3)]
    all_chamfer      = [[] for _ in range(3)]

    for batch in tqdm(dataset, desc="Evaluating Model"):
        terrain, fire_seq, tnh_seq, wind_seq = batch

        # 예측 (B=1 가정)
        mean_preds = get_ensemble_predictions(generator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble)
        target_seq = fire_seq[:, 1:4, ...]  # (B,3,H,W,1) -> GT for 4h,8h,12h

        for t in range(3):
            pred_t   = mean_preds[:, t, ...]   # (B,H,W,1)
            target_t = target_seq[:, t, ...]   # (B,H,W,1)

            # ------------------------------
            # 2-1. 불 번짐 영역 기반 지표들
            # ------------------------------
            mask_t = get_mask(target_t, pred_t, threshold=thr_mask, mode="union")  # (B,H,W,1)

            mse_t = masked_mse(target_t, pred_t, mask_t).numpy()
            if not np.isnan(mse_t):
                all_mse_scores[t].append(float(mse_t))

            ssim_t = masked_ssim(target_t, pred_t, mask_t, max_val=1.0).numpy()
            if not np.isnan(ssim_t):
                all_ssim_scores[t].append(float(ssim_t))

            iou_t = calculate_iou(target_t, pred_t, threshold=thr_mask).numpy()
            all_iou_scores[t].append(float(iou_t))

            # ------------------------------
            # 2-2. 화선(경계선) 기반 지표들
            # ------------------------------
            # (A) 이 시점까지 "불이 번진 영역" 이진화
            #     - thr_mask 기준으로 binary burn map 얻기
            area_true_bin = get_fire_area_binary(_to_hw(target_t), threshold=thr_mask)  # (H,W) 0/1
            area_pred_bin = get_fire_area_binary(_to_hw(pred_t),   threshold=thr_mask)  # (H,W) 0/1

            # (B) 경계선(화선) 추출
            edge_true = get_fireline_edge(area_true_bin)  # (H,W) 0/1
            edge_pred = get_fireline_edge(area_pred_bin)  # (H,W) 0/1

            # (C) 화선 길이 비교 -> PerimeterRelativeError
            L_true = fireline_length(edge_true, pixel_size=pixel_size)
            L_pred = fireline_length(edge_pred, pixel_size=pixel_size)

            # L_true가 0이면(=아직 화선이 없다면) 복잡도 비교 자체가 의미 없음 -> NaN
            perim_relerr_t = np.nan
            if float(L_true.numpy()) > 0.0:
                perim_relerr_t = tf.abs(L_true - L_pred) / (L_true + 1e-7)
                perim_relerr_t = float(perim_relerr_t.numpy())

            if not np.isnan(perim_relerr_t):
                all_perim_relerr[t].append(perim_relerr_t)

            # (D) Boundary IoU (화선끼리의 IoU)
            biou_t = boundary_iou(edge_true, edge_pred).numpy()
            # edge가 둘 다 완전 비어있으면 biou_t는 NaN 비슷하게 해야할 수도 있는데
            # boundary_iou() 안에서는 0/0 -> NaN 안나오게 1e-7 더했으므로 그냥 append
            all_biou_scores[t].append(float(biou_t))

            # (E) Chamfer Distance (화선 위치 평균 거리)
            chamfer_t = chamfer_distance(edge_true, edge_pred).numpy()
            if not np.isnan(chamfer_t):
                all_chamfer[t].append(float(chamfer_t))

    # 평균 산출
    avg_mse   = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_mse_scores]
    avg_ssim  = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_ssim_scores]
    avg_iou   = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_iou_scores]

    avg_perim = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_perim_relerr]
    avg_biou  = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_biou_scores]
    avg_chamf = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_chamfer]

    return {
        "MSE": avg_mse,                        # 낮을수록 좋음
        "SSIM": avg_ssim,                      # 높을수록 좋음
        "IoU": avg_iou,                        # 높을수록 좋음
        "PerimeterRelErr": avg_perim,          # 낮을수록 좋음 (둘레 복잡도 잘 맞춤)
        "BoundaryIoU": avg_biou,               # 높을수록 좋음 (화선 위치 맞춤)
        "ChamferDist": avg_chamf               # 낮을수록 좋음 (화선 평균 거리 오차)
    }


# =========================================================
# 3. 데이터셋 로드/평가/시각화
# =========================================================

print("Loading datasets...")
train_tfrecord = "./train_mid_128_all.tfrecord"
train_dataset = tf.data.TFRecordDataset([train_tfrecord])
train_dataset = train_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1, drop_remainder=True)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
#train_dataset = train_dataset.take(1000)

test_tfrecord = "./test_big_128_all.tfrecord"
test_dataset = tf.data.TFRecordDataset([test_tfrecord])
test_dataset = test_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1, drop_remainder=True)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
#test_dataset = test_dataset.take(1000)
print("Datasets loaded.")

print("\nEvaluating on Test Dataset...")
test_results = evaluate_model(generator, test_dataset, n_ensemble=3, thr_mask=0.1, pixel_size=1.0)
print("Test:", test_results)

print("\nEvaluating on Train Dataset...")
train_results = evaluate_model(generator, train_dataset, n_ensemble=3, thr_mask=0.1, pixel_size=1.0)
print("Train:", train_results)

print("\n--- Evaluation Results ---")


# -----------------
# 시각화 함수 확장
# -----------------
def plot_results(train_res, test_res, model_name="CGAN"):
    labels = ['4h', '8h', '12h']
    x = np.arange(len(labels))
    width = 0.35

    fig, axes = plt.subplots(2, 3, figsize=(28, 12))
    fig.suptitle(f'{model_name} Model Evaluation', fontsize=18)

    metric_names = [
        ("MSE", "Mean Squared Error (↓ better)"),
        ("SSIM", "Structural Similarity (↑ better)"),
        ("IoU", "Burned Area IoU (↑ better)"),
        ("PerimeterRelErr", "Perimeter Relative Error (↓ better)"),
        ("BoundaryIoU", "Boundary IoU (↑ better)"),
        ("ChamferDist", "Chamfer Distance (px, ↓ better)")
    ]

    for ax, (key, title) in zip(axes.flatten(), metric_names):
        train_vals = train_res[key]
        test_vals  = test_res[key]

        rects1 = ax.bar(x - width/2, train_vals, width, label='Train')
        rects2 = ax.bar(x + width/2, test_vals,  width, label='Test')

        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        ax.legend()

        # 막대 위 숫자 라벨
        ax.bar_label(rects1, padding=3, fmt='%.4f')
        ax.bar_label(rects2, padding=3, fmt='%.4f')

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


print("\nPlotting results...")
plot_results(train_results, test_results, model_name="CGAN")


In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm

# -----------------
# 유틸 함수들
# -----------------

def get_mask(y_true, y_pred, threshold=0.1, mode="union"):
    """
    불난 영역 마스크를 만든다.
    mode:
      - "gt":    GT에서 불난 영역만 (y_true > thr)
      - "pred":  예측에서 불났다고 한 영역만 (y_pred > thr)
      - "union": 둘 중 하나라도 불났다고 한 영역 (OR)
      - "inter": 둘 다 불났다고 한 영역 (AND)

    wildfire 평가는 보통 "union"을 많이 쓴다.
    이유: 모델이 좀 더 크게 혹은 작게 예측했어도
         관심영역(불 가능성 있는 곳) 전체에서 비교하고 싶기 때문.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    if mode == "gt":
        mask = y_true_bin
    elif mode == "pred":
        mask = y_pred_bin
    elif mode == "inter":
        mask = y_true_bin * y_pred_bin
    else:  # "union"
        mask = tf.cast((y_true_bin + y_pred_bin) > 0.0, tf.float32)

    return mask  # same shape as y_true, {0,1}


def masked_mse(y_true, y_pred, mask):
    """
    마스크 영역 내에서만 MSE 계산.
    mask: 0/1 same shape as y_true
    """
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)
    # 해당 시간스텝에 불난 픽셀이 전혀 없는 경우(denom=0) -> None 리턴해서 스킵하게 하자
    mse_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_sum(diff2) / denom
    )
    return mse_val


def masked_ssim(y_true, y_pred, mask, max_val=1.0):
    """
    SSIM을 마스크 영역에서만 계산하고 싶으면 약간의 트릭이 필요해.
    tf.image.ssim()은 전체 이미지를 본다.
    그래서 우리는 마스크 바깥은 전부 GT와 Pred를 똑같이 만들어서
    SSIM 점수에 영향을 거의 못 주게 할 수 있다.

    아이디어:
      - 마스크 밖은 y_true_masked = y_pred_masked = 0 (같은 값) 으로 맞춘다.
      - 마스크 안은 실제 값 유지.
    이렇게 하면 SSIM은 사실상 마스크 안 구조 차이만 민감하게 본다.
    """

    y_true_focus = y_true * mask
    y_pred_focus = y_pred * mask

    # 마스크 밖은 동일한 값으로 채워서 '차이 없음'으로 처리
    # 여기서는 0으로 채움
    # (이미 y_true_focus / y_pred_focus는 밖이 0이라 동일함)
    ssim_val = tf.image.ssim(y_true_focus, y_pred_focus, max_val=max_val)

    # 만약 mask가 전부 0이면? -> NaN으로 처리해서 스킵
    denom = tf.reduce_sum(mask)
    ssim_val = tf.cond(
        tf.equal(denom, 0.0),
        lambda: tf.constant(np.nan, dtype=tf.float32),
        lambda: tf.reduce_mean(ssim_val)
    )
    return ssim_val


def calculate_iou(y_true, y_pred, threshold=0.1):
    """
    네가 준 IoU 함수. (batch 단위 평균을 위해 약간 수정할거야)
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou


# -----------------
# 예측 함수 (네 그대로)
# -----------------
def get_ensemble_predictions(generator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=9):
    """
    하나의 샘플에 대해 자기회귀 및 앙상블 예측을 수행합니다.
    """
    initial_fire_input = fire_seq[:, 0, ...]  # (B,H,W,1)
    mean_predictions = []
    current_fire_input = initial_fire_input

    for t in range(3):  # predict 4h,8h,12h
        step_indices = tf.one_hot(t, depth=3)           # (3,)
        step_indices = tf.expand_dims(step_indices, 0)  # (1,3)
        tnh_t = tnh_seq[:, t]        # (B,4,2) or whatever shape per step
        wind_t = wind_seq[:, t]      # same idea

        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM))
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0)  # (n_ens,B,H,W,1) but B=1
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0)       # (B,H,W,1)
        mean_predictions.append(tf.expand_dims(mean_pred, axis=0))# (1,1,H,W,1)
        current_fire_input = tf.expand_dims(mean_pred, axis=0)

    return tf.stack(mean_predictions, axis=1)  # (B,3,H,W,1)

def evaluate_model(generator, dataset, n_ensemble=9, thr_mask=0.1):
    """
    데이터셋 전체에 대해 '불난 영역 기준' MSE / SSIM / IoU 계산
    """
    all_mse_scores = [[] for _ in range(3)]   # [4h_scores, 8h_scores, 12h_scores]
    all_ssim_scores = [[] for _ in range(3)]
    all_iou_scores = [[] for _ in range(3)]

    for batch in tqdm(dataset, desc="Evaluating Model"):
        # 너 데이터 구조에 맞춰서 가져오기
        terrain, fire_seq, tnh_seq, wind_seq = batch

        # 모델 예측
        mean_preds = get_ensemble_predictions(generator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble)
        # 정답 시퀀스 (4h,8h,12h)
        target_seq = fire_seq[:, 1:4, ...]  # (B,3,H,W,1)

        for t in range(3):
            pred_t   = mean_preds[:, t, ...]    # (B,H,W,1)
            target_t = target_seq[:, t, ...]    # (B,H,W,1)

            # 마스크 만들기 (batch=1 가정)
            mask_t = get_mask(target_t, pred_t, threshold=thr_mask, mode="union")  # (B,H,W,1)

            # --- MSE (마스크 내부만)
            mse_t = masked_mse(target_t, pred_t, mask_t).numpy()
            if not np.isnan(mse_t):
                all_mse_scores[t].append(float(mse_t))

            # --- SSIM (마스크 내부만 강조)
            ssim_t = masked_ssim(target_t, pred_t, mask_t, max_val=1.0).numpy()
            if not np.isnan(ssim_t):
                all_ssim_scores[t].append(float(ssim_t))

            # --- IoU (이건 불 영역기반이라 그대로)
            iou_t = calculate_iou(target_t, pred_t, threshold=thr_mask).numpy()
            all_iou_scores[t].append(float(iou_t))

    # 각 시간대별 평균 점수 계산
    avg_mse  = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_mse_scores]
    avg_ssim = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_ssim_scores]
    avg_iou  = [np.mean(scores) if len(scores)>0 else np.nan for scores in all_iou_scores]

    return {"MSE": avg_mse, "SSIM": avg_ssim, "IoU": avg_iou}

# --- 3. 메인 실행 및 시각화 ---

# 학습된 Generator 모델 로드 (예시)
# gen = tf.keras.models.load_model('path/to/your/generator_model')

# 데이터셋 로드
print("Loading datasets...")
train_tfrecord = "./ignition/train_mid_128_all.tfrecord"
train_dataset = tf.data.TFRecordDataset([train_tfrecord])
train_dataset = train_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
train_dataset = train_dataset.take(500)


test_tfrecord = "./ignition/test_mid_128_all.tfrecord"
test_dataset = tf.data.TFRecordDataset([test_tfrecord])
test_dataset = test_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.take(500)
print("Datasets loaded.")

# 모델 평가 실행
print("\nEvaluating on Test Dataset...")
test_results = evaluate_model(generator, test_dataset)
print(f"Test Dataset  -> MSE: {test_results['MSE']}, SSIM: {test_results['SSIM']}")
print("\nEvaluating on Train Dataset...")
train_results = evaluate_model(generator, train_dataset)
print(f"Train Dataset -> MSE: {train_results['MSE']}, SSIM: {train_results['SSIM']}")
# 결과 출력
print("\n--- Evaluation Results ---")




# --- 시각화 ---

def plot_results(train_res, test_res, model_name=""):
    labels = ['4h', '8h', '12h']
    x = np.arange(len(labels))
    width = 0.35

    # subplots를 1x2에서 1x3으로 변경하고, figsize 너비를 늘립니다.
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 6))
    fig.suptitle(f'{model_name} Model Evaluation', fontsize=16)

    # MSE 그래프 (기존과 동일)
    rects1 = ax1.bar(x - width/2, train_res['MSE'], width, label='Train')
    rects2 = ax1.bar(x + width/2, test_res['MSE'], width, label='Test')
    ax1.set_ylabel('Mean Squared Error (MSE)')
    ax1.set_title('Average MSE (Lower is Better)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.bar_label(rects1, padding=3, fmt='%.4f')
    ax1.bar_label(rects2, padding=3, fmt='%.4f')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # SSIM 그래프 (기존과 동일)
    rects3 = ax2.bar(x - width/2, train_res['SSIM'], width, label='Train')
    rects4 = ax2.bar(x + width/2, test_res['SSIM'], width, label='Test')
    ax2.set_ylabel('Structural Similarity (SSIM)')
    ax2.set_title('Average SSIM (Higher is Better)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.legend()
    ax2.bar_label(rects3, padding=3, fmt='%.4f')
    ax2.bar_label(rects4, padding=3, fmt='%.4f')
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    
    # --- IoU 그래프 추가 ---
    rects5 = ax3.bar(x - width/2, train_res['IoU'], width, label='Train')
    rects6 = ax3.bar(x + width/2, test_res['IoU'], width, label='Test')
    ax3.set_ylabel('Intersection over Union (IoU)')
    ax3.set_title('Average IoU (Higher is Better)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels)
    ax3.legend()
    ax3.bar_label(rects5, padding=3, fmt='%.4f')
    ax3.bar_label(rects6, padding=3, fmt='%.4f')
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    # --------------------
    
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print("\nPlotting results...")
plot_results(train_results, test_results)
           

In [ ]:
# 모델 평가 실행
print("\nEvaluating on Test Dataset...")
print(f"Test Dataset  -> MSE: {test_results['MSE']}, SSIM: {test_results['SSIM']} , IoU: {test_results['IoU']}")
print("\nEvaluating on Train Dataset...")
print(f"Train Dataset -> MSE: {train_results['MSE']}, SSIM: {train_results['SSIM']},, IoU: {train_results['IoU']}")

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from scipy.linalg import sqrtm

# --- FID 계산을 위한 함수 (새로 추가) ---
def calculate_fid(model, real_images, generated_images):
    """
    실제 이미지와 생성 이미지 세트 간의 FID 점수를 계산합니다.
    이미지는 InceptionV3에 맞게 전처리됩니다.
    """
    # InceptionV3 모델 입력 형식에 맞게 이미지 전처리
    # 1. 3채널로 복제 (원본이 1채널이므로)
    # 2. 크기 조정 (최소 75x75 필요)
    # 3. Keras의 전처리 함수 적용
    def preprocess_images(images):
        # images: (N, H, W, 1)
        images_resized = tf.image.resize(images, [75, 75]) # InceptionV3 최소 크기
        images_rgb = tf.image.grayscale_to_rgb(images_resized) # 3채널로 복제
        images_preprocessed = preprocess_input(images_rgb)
        return images_preprocessed

    real_processed = preprocess_images(real_images)
    gen_processed = preprocess_images(generated_images)

    # InceptionV3 모델을 통해 특징 추출
    real_features = model.predict(real_processed)
    gen_features = model.predict(gen_processed)

    # 각 특징 벡터의 평균과 공분산 계산
    mu_real, sigma_real = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu_gen, sigma_gen = np.mean(gen_features, axis=0), np.cov(gen_features, rowvar=False)

    # FID 점수 계산
    ssdiff = np.sum((mu_real - mu_gen)**2.0)
    covmean = sqrtm(sigma_real.dot(sigma_gen))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma_real + sigma_gen - 2.0 * covmean)
    return fid

# --- IoU 계산을 위한 함수 (새로 추가) ---
def calculate_iou(y_true, y_pred, threshold=0.1):
    """
    임계값을 기준으로 이진화한 후 IoU를 계산합니다.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7) # 0으로 나누는 것을 방지
    return iou.numpy().item()



def ensemble_chaining_with_disc_scores(generator, discriminator, 
                                     terrain, fire_seq, tnh_seq, wind_seq, 
                                     n_ensemble=9):
    """
    앙상블 체이닝 추론을 수행하며, 각 단계별 판별자 점수를 함께 반환합니다. (Shape 오류 수정 버전)
    """
    # fire_seq: (batch, time_steps, H, W, C) -> fire_seq[:, 0, ...]는 (batch, H, W, C)
    initial_fire_input = fire_seq[:, 0, ...] # 0h (현재) fire state
    
    mean_predictions = []
    variance_predictions = []
    real_scores = []
    fake_scores = []
    
    current_fire_input = initial_fire_input # (batch, H, W, C)
    
    # 4h, 8h, 12h 예측 (총 3 스텝)
    # tnh_seq.shape[1]은 4 (0h, 4h, 8h, 12h) 이므로 range(4)가 될 것.
    # 하지만 예측은 4h, 8h, 12h (3스텝)이므로 t+1을 사용하기 위해 range(3)으로 제한합니다.
    for t in range(3): # 0, 1, 2 에 해당 (각각 4h, 8h, 12h 예측)
        step_indices = tf.one_hot(t, depth=3) # t는 0, 1, 2
        step_indices = tf.expand_dims(step_indices, axis=0) # (1, 3)

        # tnh_seq와 wind_seq에서 해당 시간 스텝 데이터 추출
        # tnh_seq: (batch, 4, 1, 2) -> tnh_seq[:, t+1, ...] 는 (batch, 1, 2)
        # wind_seq: (batch, 4, 2, 1) -> wind_seq[:, t+1, ...] 는 (batch, 2, 1)
        # 예측은 fire_seq의 t+1 스텝 (4h, 8h, 12h)의 날씨 정보를 사용해야 함.
        tnh_t = tnh_seq[:, t]  # 0h 인덱스가 아니라 예측하려는 스텝의 날씨 정보
        wind_t = wind_seq[:, t]
        
        # --- 앙상블 예측 ---
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM)) # 배치 크기 1에 맞춰 (1, LATENT_DIM)
            
            # 제너레이터 입력: [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise]
            # current_fire_input: (1, H, W, C)
            # terrain: (1, H, W, C)
            # tnh_t: (1, 1, 2)
            # wind_t: (1, 2, 1)
            # step_indices: (1, 3)
            # noise: (1, LATENT_DIM)
            
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0) # (n_ensemble, H, W, C)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0) # (H, W, C)
        variance_pred = tf.math.reduce_variance(ensemble_tensor, axis=0) # (H, W, C)

        mean_predictions.append(tf.expand_dims(mean_pred, axis=0)) # (1, H, W, C) -> 각 스텝 예측 저장
        variance_predictions.append(tf.expand_dims(variance_pred, axis=0)) # (1, H, W, C)

        # --- 판별자 점수 계산 ---
        mean_pred_batched = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C) - 판별자 입력을 위해 배치 차원 추가
        
        # true_next_fire: fire_seq에서 예측하려는 다음 실제 불 이미지 (fire_seq[:, t+1, ...])
        true_next_fire = fire_seq[:, t + 1, ...] # (1, H, W, C)
        
        disc_inputs_real = [true_next_fire, terrain, tnh_t, wind_t, step_indices]
        disc_inputs_fake = [mean_pred_batched, terrain, tnh_t, wind_t, step_indices]

        # discriminator는 여러 개의 패치에 대한 점수를 반환할 수 있으므로, 평균을 취함
        real_score_patches = discriminator(disc_inputs_real, training=False) # (1, num_patches) 또는 (1, 1)
        fake_score_patches = discriminator(disc_inputs_fake, training=False) # (1, num_patches) 또는 (1, 1)
        
        avg_real_score = tf.reduce_mean(real_score_patches) # 스칼라
        avg_fake_score = tf.reduce_mean(fake_score_patches) # 스칼라
        
        real_scores.append(avg_real_score)
        fake_scores.append(avg_fake_score)
        
        # 다음 스텝의 입력으로 현재 예측 결과를 사용
        # 현재 mean_pred는 (H, W, C) 이므로, current_fire_input (1, H, W, C)로 만들어야 함
        current_fire_input = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C)

    # 결과를 쌓아서 반환
    # mean_predictions: 각 요소가 (1, H, W, C) -> stacked_means: (3, 1, H, W, C) -> tf.stack(axis=1)하면 (1, 3, H, W, C)
    stacked_means = tf.stack(mean_predictions, axis=1) # (batch, num_steps, H, W, C)
    stacked_vars = tf.stack(variance_predictions, axis=1) # (batch, num_steps, H, W, C)
    
    # real_scores/fake_scores는 각 스텝별 스칼라 텐서 리스트 (3개)
    return stacked_means, stacked_vars, real_scores, fake_scores

# --- 메인 코드 (수정 불필요) ---

# 테스트 데이터셋 불러오기
#test_tfrecord = "./ignition/test_mid_128_all.tfrecord"
test_tfrecord = "./ignition/test_128_56ea.tfrecord"
test_dataset = tf.data.TFRecordDataset([test_tfrecord])
test_dataset = test_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
test_list = list(test_dataset)
num_batches = len(test_list)
print(len(test_list))
# 원하는 샘플 개수 선택
# --- 원하는 샘플 개수와 시작 인덱스 설정 ---
NN = 0 # 0부터 시작, 0이면 0~35, 1이면 36~71 등 36개 단위로 이동
start_sample_idx = 28 * NN # 0부터 시작하여 시각화를 시작할 샘플의 인덱스
num_samples_to_visualize = 28 # 원하는 샘플 수 (예: 1~36)

# 인덱스 유효성 검사 및 조정
if start_sample_idx >= num_batches:
    print(f"시작 인덱스 {start_sample_idx}가 전체 샘플 수 {num_batches}보다 크거나 같습니다. 시각화할 샘플이 없습니다.")
    exit()

if start_sample_idx + num_samples_to_visualize > num_batches:
    num_samples_to_visualize = num_batches - start_sample_idx
    print(f"경고: 요청된 샘플 수가 전체 데이터셋을 초과합니다. {num_samples_to_visualize}개의 샘플만 시각화합니다.")

if num_samples_to_visualize <= 0:
    print("시각화할 샘플이 없습니다. 시작 인덱스와 샘플 수를 확인하세요.")
    exit()

# 순차적인 인덱스 리스트 생성
sequential_batch_idxs = list(range(start_sample_idx, start_sample_idx + num_samples_to_visualize))

inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(75, 75, 3))
all_gt_images_cgan = []
all_pred_images_cgan = []

# --- 시각화 설정 ---
# 각 샘플은 2행 (실제, 예측) x 3열 (4h, 8h, 12h)을 차지합니다.
num_cols_per_sample_row = 3 # 4h, 8h, 12h
num_rows_per_sample = 2 # 실제 이미지, 예측 평균 이미지

total_plot_cols = num_cols_per_sample_row # 전체 subplot 그리드의 열 수 (3)
total_plot_rows = num_samples_to_visualize * num_rows_per_sample # 전체 subplot 그리드의 행 수 (샘플 수 * 2)

# Figure 크기 조정
# 각 서브플롯당 대략적인 크기를 5x5 인치로 가정하여 전체 Figure 크기를 계산합니다.
plt.figure(figsize=(total_plot_cols * 5, total_plot_rows * 5)) 

print(f"\n총 {num_samples_to_visualize}개의 샘플을 시각화합니다 (인덱스 {start_sample_idx}부터).")
print(f"생성될 시각화 그리드: {total_plot_rows} 행 x {total_plot_cols} 열")

for i, current_sample_relative_idx in enumerate(range(num_samples_to_visualize)):
    bidx = sequential_batch_idxs[current_sample_relative_idx]
    terrain, fire_seq, tnh_seq, wind_seq = test_list[bidx]

    print(f"\n--- Processing Sample Index: {bidx} ---")

    start_time = time.time()
    mean_preds, variance_preds, real_scores, fake_scores = ensemble_chaining_with_disc_scores(
        generator, discriminator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=6
    )
    end_time = time.time()
    prediction_time = end_time - start_time
    print(f"Prediction Time for Sample {bidx}: {prediction_time:.2f} seconds")

    mean_preds_np = mean_preds.numpy()
    target_seq_np = fire_seq[0, 1:4].numpy()
    real_scores_np = np.array([score.numpy() for score in real_scores])
    fake_scores_np = np.array([score.numpy() for score in fake_scores])

    # FID 계산을 위해 현재 배치의 이미지들 저장
    all_gt_images_cgan.extend(target_seq_np) # (3, H, W, C) 이미지가 추가됨
    all_pred_images_cgan.extend(mean_preds_np[0]) # (3, H, W, C) 이미지가 추가됨

    num_steps = mean_preds_np.shape[1]
    hours = [4, 8, 12]

    current_sample_start_row = i * 2

    for t in range(num_steps):
        gt_img_tensor = tf.convert_to_tensor(target_seq_np[t], dtype=tf.float32)
        mean_img_tensor = tf.convert_to_tensor(mean_preds_np[0, t], dtype=tf.float32)

        # MSE 계산
        mse_loss = tf.keras.losses.MeanSquaredError()
        mse_score = mse_loss(gt_img_tensor, mean_img_tensor).numpy().item()

        # SSIM 계산
        ssim_score = tf.image.ssim(
            tf.expand_dims(gt_img_tensor, axis=0),
            tf.expand_dims(mean_img_tensor, axis=0),
            max_val=1.0
        ).numpy().item()

        # IoU 계산 (새로 추가)
        iou_score = calculate_iou(gt_img_tensor, mean_img_tensor, threshold=0.1)

        gt_img = np.squeeze(gt_img_tensor.numpy())
        mean_img = np.squeeze(mean_img_tensor.numpy())

        # 실제 이미지 subplot
        plt.subplot(total_plot_rows, total_plot_cols, current_sample_start_row * total_plot_cols + t + 1)
        plt.imshow(gt_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"True {hours[t]}h (Idx {bidx})\n(Score: {real_scores_np[t]:.3f})")
        plt.axis('off')

        # 예측 이미지 subplot (제목에 IoU 추가)
        plt.subplot(total_plot_rows, total_plot_cols, (current_sample_start_row + 1) * total_plot_cols + t + 1)
        plt.imshow(mean_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"Pred Mean {hours[t]}h (Idx {bidx})\n"
                  f"(D-Score: {fake_scores_np[t]:.3f}, MSE: {mse_score:.4f}\n"
                  f"SSIM: {ssim_score:.4f}, IoU: {iou_score:.4f})") # IoU 추가
        plt.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# --- FID 점수 계산 및 출력 (루프 종료 후) ---
print("\n--- Calculating FID Score for CGAN Model ---")
# 리스트를 텐서로 변환 (N, H, W, C)
all_gt_tensor_cgan = tf.convert_to_tensor(all_gt_images_cgan, dtype=tf.float32)
all_pred_tensor_cgan = tf.convert_to_tensor(all_pred_images_cgan, dtype=tf.float32)

# FID 계산
fid_score_cgan = calculate_fid(inception_model, all_gt_tensor_cgan, all_pred_tensor_cgan)
print(f"FID Score (CGAN vs. Ground Truth): {fid_score_cgan:.4f}")

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from scipy.linalg import sqrtm

# --- FID 계산을 위한 함수 (새로 추가) ---
def calculate_fid(model, real_images, generated_images):
    """
    실제 이미지와 생성 이미지 세트 간의 FID 점수를 계산합니다.
    이미지는 InceptionV3에 맞게 전처리됩니다.
    """
    # InceptionV3 모델 입력 형식에 맞게 이미지 전처리
    # 1. 3채널로 복제 (원본이 1채널이므로)
    # 2. 크기 조정 (최소 75x75 필요)
    # 3. Keras의 전처리 함수 적용
    def preprocess_images(images):
        # images: (N, H, W, 1)
        images_resized = tf.image.resize(images, [75, 75]) # InceptionV3 최소 크기
        images_rgb = tf.image.grayscale_to_rgb(images_resized) # 3채널로 복제
        images_preprocessed = preprocess_input(images_rgb)
        return images_preprocessed

    real_processed = preprocess_images(real_images)
    gen_processed = preprocess_images(generated_images)

    # InceptionV3 모델을 통해 특징 추출
    real_features = model.predict(real_processed)
    gen_features = model.predict(gen_processed)

    # 각 특징 벡터의 평균과 공분산 계산
    mu_real, sigma_real = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu_gen, sigma_gen = np.mean(gen_features, axis=0), np.cov(gen_features, rowvar=False)

    # FID 점수 계산
    ssdiff = np.sum((mu_real - mu_gen)**2.0)
    covmean = sqrtm(sigma_real.dot(sigma_gen))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma_real + sigma_gen - 2.0 * covmean)
    return fid

# --- IoU 계산을 위한 함수 (새로 추가) ---
def calculate_iou(y_true, y_pred, threshold=0.1):
    """
    임계값을 기준으로 이진화한 후 IoU를 계산합니다.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7) # 0으로 나누는 것을 방지
    return iou.numpy().item()



def ensemble_chaining_with_disc_scores(generator, discriminator, 
                                     terrain, fire_seq, tnh_seq, wind_seq, 
                                     n_ensemble=9):
    """
    앙상블 체이닝 추론을 수행하며, 각 단계별 판별자 점수를 함께 반환합니다. (Shape 오류 수정 버전)
    """
    # fire_seq: (batch, time_steps, H, W, C) -> fire_seq[:, 0, ...]는 (batch, H, W, C)
    initial_fire_input = fire_seq[:, 0, ...] # 0h (현재) fire state
    
    mean_predictions = []
    variance_predictions = []
    real_scores = []
    fake_scores = []
    
    current_fire_input = initial_fire_input # (batch, H, W, C)
    
    # 4h, 8h, 12h 예측 (총 3 스텝)
    # tnh_seq.shape[1]은 4 (0h, 4h, 8h, 12h) 이므로 range(4)가 될 것.
    # 하지만 예측은 4h, 8h, 12h (3스텝)이므로 t+1을 사용하기 위해 range(3)으로 제한합니다.
    for t in range(3): # 0, 1, 2 에 해당 (각각 4h, 8h, 12h 예측)
        step_indices = tf.one_hot(t, depth=3) # t는 0, 1, 2
        step_indices = tf.expand_dims(step_indices, axis=0) # (1, 3)

        # tnh_seq와 wind_seq에서 해당 시간 스텝 데이터 추출
        # tnh_seq: (batch, 4, 1, 2) -> tnh_seq[:, t+1, ...] 는 (batch, 1, 2)
        # wind_seq: (batch, 4, 2, 1) -> wind_seq[:, t+1, ...] 는 (batch, 2, 1)
        # 예측은 fire_seq의 t+1 스텝 (4h, 8h, 12h)의 날씨 정보를 사용해야 함.
        tnh_t = tnh_seq[:, t]  # 0h 인덱스가 아니라 예측하려는 스텝의 날씨 정보
        wind_t = wind_seq[:, t]
        
        # --- 앙상블 예측 ---
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM)) # 배치 크기 1에 맞춰 (1, LATENT_DIM)
            
            # 제너레이터 입력: [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise]
            # current_fire_input: (1, H, W, C)
            # terrain: (1, H, W, C)
            # tnh_t: (1, 1, 2)
            # wind_t: (1, 2, 1)
            # step_indices: (1, 3)
            # noise: (1, LATENT_DIM)
            
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0) # (n_ensemble, H, W, C)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0) # (H, W, C)
        variance_pred = tf.math.reduce_variance(ensemble_tensor, axis=0) # (H, W, C)

        mean_predictions.append(tf.expand_dims(mean_pred, axis=0)) # (1, H, W, C) -> 각 스텝 예측 저장
        variance_predictions.append(tf.expand_dims(variance_pred, axis=0)) # (1, H, W, C)

        # --- 판별자 점수 계산 ---
        mean_pred_batched = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C) - 판별자 입력을 위해 배치 차원 추가
        
        # true_next_fire: fire_seq에서 예측하려는 다음 실제 불 이미지 (fire_seq[:, t+1, ...])
        true_next_fire = fire_seq[:, t + 1, ...] # (1, H, W, C)
        
        disc_inputs_real = [true_next_fire, terrain, tnh_t, wind_t, step_indices]
        disc_inputs_fake = [mean_pred_batched, terrain, tnh_t, wind_t, step_indices]

        # discriminator는 여러 개의 패치에 대한 점수를 반환할 수 있으므로, 평균을 취함
        real_score_patches = discriminator(disc_inputs_real, training=False) # (1, num_patches) 또는 (1, 1)
        fake_score_patches = discriminator(disc_inputs_fake, training=False) # (1, num_patches) 또는 (1, 1)
        
        avg_real_score = tf.reduce_mean(real_score_patches) # 스칼라
        avg_fake_score = tf.reduce_mean(fake_score_patches) # 스칼라
        
        real_scores.append(avg_real_score)
        fake_scores.append(avg_fake_score)
        
        # 다음 스텝의 입력으로 현재 예측 결과를 사용
        # 현재 mean_pred는 (H, W, C) 이므로, current_fire_input (1, H, W, C)로 만들어야 함
        current_fire_input = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C)

    # 결과를 쌓아서 반환
    # mean_predictions: 각 요소가 (1, H, W, C) -> stacked_means: (3, 1, H, W, C) -> tf.stack(axis=1)하면 (1, 3, H, W, C)
    stacked_means = tf.stack(mean_predictions, axis=1) # (batch, num_steps, H, W, C)
    stacked_vars = tf.stack(variance_predictions, axis=1) # (batch, num_steps, H, W, C)
    
    # real_scores/fake_scores는 각 스텝별 스칼라 텐서 리스트 (3개)
    return stacked_means, stacked_vars, real_scores, fake_scores

# --- 메인 코드 (수정 불필요) ---

# 테스트 데이터셋 불러오기
test_tfrecord = "./ignition/test_big_128_all.tfrecord"
#test_tfrecord2 = "./ignition/test_128_56ea.tfrecord"
test_dataset2 = tf.data.TFRecordDataset([test_tfrecord])
test_dataset2 = test_dataset2.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_dataset2 = test_dataset2.prefetch(tf.data.AUTOTUNE)
test_list = list(test_dataset2)
num_batches = len(test_list)
print(len(test_list))
# 원하는 샘플 개수 선택
# --- 원하는 샘플 개수와 시작 인덱스 설정 ---
NN = 1 # 0부터 시작, 0이면 0~35, 1이면 36~71 등 36개 단위로 이동
start_sample_idx = 28 * NN # 0부터 시작하여 시각화를 시작할 샘플의 인덱스
num_samples_to_visualize = 28 # 원하는 샘플 수 (예: 1~36)

# 인덱스 유효성 검사 및 조정
if start_sample_idx >= num_batches:
    print(f"시작 인덱스 {start_sample_idx}가 전체 샘플 수 {num_batches}보다 크거나 같습니다. 시각화할 샘플이 없습니다.")
    exit()

if start_sample_idx + num_samples_to_visualize > num_batches:
    num_samples_to_visualize = num_batches - start_sample_idx
    print(f"경고: 요청된 샘플 수가 전체 데이터셋을 초과합니다. {num_samples_to_visualize}개의 샘플만 시각화합니다.")

if num_samples_to_visualize <= 0:
    print("시각화할 샘플이 없습니다. 시작 인덱스와 샘플 수를 확인하세요.")
    exit()

# 순차적인 인덱스 리스트 생성
sequential_batch_idxs = list(range(start_sample_idx, start_sample_idx + num_samples_to_visualize))

inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(75, 75, 3))
all_gt_images_cgan = []
all_pred_images_cgan = []

# --- 시각화 설정 ---
# 각 샘플은 2행 (실제, 예측) x 3열 (4h, 8h, 12h)을 차지합니다.
num_cols_per_sample_row = 3 # 4h, 8h, 12h
num_rows_per_sample = 2 # 실제 이미지, 예측 평균 이미지

total_plot_cols = num_cols_per_sample_row # 전체 subplot 그리드의 열 수 (3)
total_plot_rows = num_samples_to_visualize * num_rows_per_sample # 전체 subplot 그리드의 행 수 (샘플 수 * 2)

# Figure 크기 조정
# 각 서브플롯당 대략적인 크기를 5x5 인치로 가정하여 전체 Figure 크기를 계산합니다.
plt.figure(figsize=(total_plot_cols * 5, total_plot_rows * 5)) 

print(f"\n총 {num_samples_to_visualize}개의 샘플을 시각화합니다 (인덱스 {start_sample_idx}부터).")
print(f"생성될 시각화 그리드: {total_plot_rows} 행 x {total_plot_cols} 열")

for i, current_sample_relative_idx in enumerate(range(num_samples_to_visualize)):
    bidx = sequential_batch_idxs[current_sample_relative_idx]
    terrain, fire_seq, tnh_seq, wind_seq = test_list[bidx]

    print(f"\n--- Processing Sample Index: {bidx} ---")

    start_time = time.time()
    mean_preds, variance_preds, real_scores, fake_scores = ensemble_chaining_with_disc_scores(
        gen, disc, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=9
    )
    end_time = time.time()
    prediction_time = end_time - start_time
    print(f"Prediction Time for Sample {bidx}: {prediction_time:.2f} seconds")

    mean_preds_np = mean_preds.numpy()
    target_seq_np = fire_seq[0, 1:4].numpy()
    real_scores_np = np.array([score.numpy() for score in real_scores])
    fake_scores_np = np.array([score.numpy() for score in fake_scores])

    # FID 계산을 위해 현재 배치의 이미지들 저장
    all_gt_images_cgan.extend(target_seq_np) # (3, H, W, C) 이미지가 추가됨
    all_pred_images_cgan.extend(mean_preds_np[0]) # (3, H, W, C) 이미지가 추가됨

    num_steps = mean_preds_np.shape[1]
    hours = [4, 8, 12]

    current_sample_start_row = i * 2

    for t in range(num_steps):
        gt_img_tensor = tf.convert_to_tensor(target_seq_np[t], dtype=tf.float32)
        mean_img_tensor = tf.convert_to_tensor(mean_preds_np[0, t], dtype=tf.float32)

        # MSE 계산
        mse_loss = tf.keras.losses.MeanSquaredError()
        mse_score = mse_loss(gt_img_tensor, mean_img_tensor).numpy().item()

        # SSIM 계산
        ssim_score = tf.image.ssim(
            tf.expand_dims(gt_img_tensor, axis=0),
            tf.expand_dims(mean_img_tensor, axis=0),
            max_val=1.0
        ).numpy().item()

        # IoU 계산 (새로 추가)
        iou_score = calculate_iou(gt_img_tensor, mean_img_tensor, threshold=0.1)

        gt_img = np.squeeze(gt_img_tensor.numpy())
        mean_img = np.squeeze(mean_img_tensor.numpy())

        # 실제 이미지 subplot
        plt.subplot(total_plot_rows, total_plot_cols, current_sample_start_row * total_plot_cols + t + 1)
        plt.imshow(gt_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"True {hours[t]}h (Idx {bidx})\n(Score: {real_scores_np[t]:.3f})")
        plt.axis('off')

        # 예측 이미지 subplot (제목에 IoU 추가)
        plt.subplot(total_plot_rows, total_plot_cols, (current_sample_start_row + 1) * total_plot_cols + t + 1)
        plt.imshow(mean_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"Pred Mean {hours[t]}h (Idx {bidx})\n"
                  f"(D-Score: {fake_scores_np[t]:.3f}, MSE: {mse_score:.4f}\n"
                  f"SSIM: {ssim_score:.4f}, IoU: {iou_score:.4f})") # IoU 추가
        plt.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# --- FID 점수 계산 및 출력 (루프 종료 후) ---
print("\n--- Calculating FID Score for CGAN Model ---")
# 리스트를 텐서로 변환 (N, H, W, C)
all_gt_tensor_cgan = tf.convert_to_tensor(all_gt_images_cgan, dtype=tf.float32)
all_pred_tensor_cgan = tf.convert_to_tensor(all_pred_images_cgan, dtype=tf.float32)

# FID 계산
fid_score_cgan = calculate_fid(inception_model, all_gt_tensor_cgan, all_pred_tensor_cgan)
print(f"FID Score (CGAN vs. Ground Truth): {fid_score_cgan:.4f}")

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from scipy.linalg import sqrtm

# --- FID 계산을 위한 함수 (새로 추가) ---
def calculate_fid(model, real_images, generated_images):
    """
    실제 이미지와 생성 이미지 세트 간의 FID 점수를 계산합니다.
    이미지는 InceptionV3에 맞게 전처리됩니다.
    """
    # InceptionV3 모델 입력 형식에 맞게 이미지 전처리
    # 1. 3채널로 복제 (원본이 1채널이므로)
    # 2. 크기 조정 (최소 75x75 필요)
    # 3. Keras의 전처리 함수 적용
    def preprocess_images(images):
        # images: (N, H, W, 1)
        images_resized = tf.image.resize(images, [75, 75]) # InceptionV3 최소 크기
        images_rgb = tf.image.grayscale_to_rgb(images_resized) # 3채널로 복제
        images_preprocessed = preprocess_input(images_rgb)
        return images_preprocessed

    real_processed = preprocess_images(real_images)
    gen_processed = preprocess_images(generated_images)

    # InceptionV3 모델을 통해 특징 추출
    real_features = model.predict(real_processed)
    gen_features = model.predict(gen_processed)

    # 각 특징 벡터의 평균과 공분산 계산
    mu_real, sigma_real = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu_gen, sigma_gen = np.mean(gen_features, axis=0), np.cov(gen_features, rowvar=False)

    # FID 점수 계산
    ssdiff = np.sum((mu_real - mu_gen)**2.0)
    covmean = sqrtm(sigma_real.dot(sigma_gen))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma_real + sigma_gen - 2.0 * covmean)
    return fid

# --- IoU 계산을 위한 함수 (새로 추가) ---
def calculate_iou(y_true, y_pred, threshold=0.1):
    """
    임계값을 기준으로 이진화한 후 IoU를 계산합니다.
    """
    y_true_bin = tf.cast(y_true > threshold, tf.float32)
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)

    intersection = tf.reduce_sum(y_true_bin * y_pred_bin)
    union = tf.reduce_sum(y_true_bin) + tf.reduce_sum(y_pred_bin) - intersection

    iou = (intersection + 1e-7) / (union + 1e-7) # 0으로 나누는 것을 방지
    return iou.numpy().item()



def ensemble_chaining_with_disc_scores(generator, discriminator, 
                                     terrain, fire_seq, tnh_seq, wind_seq, 
                                     n_ensemble=9):
    """
    앙상블 체이닝 추론을 수행하며, 각 단계별 판별자 점수를 함께 반환합니다. (Shape 오류 수정 버전)
    """
    # fire_seq: (batch, time_steps, H, W, C) -> fire_seq[:, 0, ...]는 (batch, H, W, C)
    initial_fire_input = fire_seq[:, 0, ...] # 0h (현재) fire state
    
    mean_predictions = []
    variance_predictions = []
    real_scores = []
    fake_scores = []
    
    current_fire_input = initial_fire_input # (batch, H, W, C)
    
    # 4h, 8h, 12h 예측 (총 3 스텝)
    # tnh_seq.shape[1]은 4 (0h, 4h, 8h, 12h) 이므로 range(4)가 될 것.
    # 하지만 예측은 4h, 8h, 12h (3스텝)이므로 t+1을 사용하기 위해 range(3)으로 제한합니다.
    for t in range(3): # 0, 1, 2 에 해당 (각각 4h, 8h, 12h 예측)
        step_indices = tf.one_hot(t, depth=3) # t는 0, 1, 2
        step_indices = tf.expand_dims(step_indices, axis=0) # (1, 3)

        # tnh_seq와 wind_seq에서 해당 시간 스텝 데이터 추출
        # tnh_seq: (batch, 4, 1, 2) -> tnh_seq[:, t+1, ...] 는 (batch, 1, 2)
        # wind_seq: (batch, 4, 2, 1) -> wind_seq[:, t+1, ...] 는 (batch, 2, 1)
        # 예측은 fire_seq의 t+1 스텝 (4h, 8h, 12h)의 날씨 정보를 사용해야 함.
        tnh_t = tnh_seq[:, t]  # 0h 인덱스가 아니라 예측하려는 스텝의 날씨 정보
        wind_t = wind_seq[:, t]
        
        # --- 앙상블 예측 ---
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM)) # 배치 크기 1에 맞춰 (1, LATENT_DIM)
            
            # 제너레이터 입력: [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise]
            # current_fire_input: (1, H, W, C)
            # terrain: (1, H, W, C)
            # tnh_t: (1, 1, 2)
            # wind_t: (1, 2, 1)
            # step_indices: (1, 3)
            # noise: (1, LATENT_DIM)
            
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0) # (n_ensemble, H, W, C)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0) # (H, W, C)
        variance_pred = tf.math.reduce_variance(ensemble_tensor, axis=0) # (H, W, C)

        mean_predictions.append(tf.expand_dims(mean_pred, axis=0)) # (1, H, W, C) -> 각 스텝 예측 저장
        variance_predictions.append(tf.expand_dims(variance_pred, axis=0)) # (1, H, W, C)

        # --- 판별자 점수 계산 ---
        mean_pred_batched = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C) - 판별자 입력을 위해 배치 차원 추가
        
        # true_next_fire: fire_seq에서 예측하려는 다음 실제 불 이미지 (fire_seq[:, t+1, ...])
        true_next_fire = fire_seq[:, t + 1, ...] # (1, H, W, C)
        
        disc_inputs_real = [true_next_fire, terrain, tnh_t, wind_t, step_indices]
        disc_inputs_fake = [mean_pred_batched, terrain, tnh_t, wind_t, step_indices]

        # discriminator는 여러 개의 패치에 대한 점수를 반환할 수 있으므로, 평균을 취함
        real_score_patches = discriminator(disc_inputs_real, training=False) # (1, num_patches) 또는 (1, 1)
        fake_score_patches = discriminator(disc_inputs_fake, training=False) # (1, num_patches) 또는 (1, 1)
        
        avg_real_score = tf.reduce_mean(real_score_patches) # 스칼라
        avg_fake_score = tf.reduce_mean(fake_score_patches) # 스칼라
        
        real_scores.append(avg_real_score)
        fake_scores.append(avg_fake_score)
        
        # 다음 스텝의 입력으로 현재 예측 결과를 사용
        # 현재 mean_pred는 (H, W, C) 이므로, current_fire_input (1, H, W, C)로 만들어야 함
        current_fire_input = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C)

    # 결과를 쌓아서 반환
    # mean_predictions: 각 요소가 (1, H, W, C) -> stacked_means: (3, 1, H, W, C) -> tf.stack(axis=1)하면 (1, 3, H, W, C)
    stacked_means = tf.stack(mean_predictions, axis=1) # (batch, num_steps, H, W, C)
    stacked_vars = tf.stack(variance_predictions, axis=1) # (batch, num_steps, H, W, C)
    
    # real_scores/fake_scores는 각 스텝별 스칼라 텐서 리스트 (3개)
    return stacked_means, stacked_vars, real_scores, fake_scores

# --- 메인 코드 (수정 불필요) ---

# 테스트 데이터셋 불러오기
#test_tfrecord = "./ignition/test_big_128_all.tfrecord"
test_tfrecord2 = "./ignition/test_128_56ea.tfrecord"
test_dataset2 = tf.data.TFRecordDataset([test_tfrecord2])
test_dataset2 = test_dataset2.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_dataset2 = test_dataset2.prefetch(tf.data.AUTOTUNE)
test_list = list(test_dataset2)
num_batches = len(test_list)
print(len(test_list))
# 원하는 샘플 개수 선택
# --- 원하는 샘플 개수와 시작 인덱스 설정 ---
NN = 1 # 0부터 시작, 0이면 0~35, 1이면 36~71 등 36개 단위로 이동
start_sample_idx = 28 * NN # 0부터 시작하여 시각화를 시작할 샘플의 인덱스
num_samples_to_visualize = 28 # 원하는 샘플 수 (예: 1~36)

# 인덱스 유효성 검사 및 조정
if start_sample_idx >= num_batches:
    print(f"시작 인덱스 {start_sample_idx}가 전체 샘플 수 {num_batches}보다 크거나 같습니다. 시각화할 샘플이 없습니다.")
    exit()

if start_sample_idx + num_samples_to_visualize > num_batches:
    num_samples_to_visualize = num_batches - start_sample_idx
    print(f"경고: 요청된 샘플 수가 전체 데이터셋을 초과합니다. {num_samples_to_visualize}개의 샘플만 시각화합니다.")

if num_samples_to_visualize <= 0:
    print("시각화할 샘플이 없습니다. 시작 인덱스와 샘플 수를 확인하세요.")
    exit()

# 순차적인 인덱스 리스트 생성
sequential_batch_idxs = list(range(start_sample_idx, start_sample_idx + num_samples_to_visualize))

inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(75, 75, 3))
all_gt_images_cgan = []
all_pred_images_cgan = []

# --- 시각화 설정 ---
# 각 샘플은 2행 (실제, 예측) x 3열 (4h, 8h, 12h)을 차지합니다.
num_cols_per_sample_row = 3 # 4h, 8h, 12h
num_rows_per_sample = 2 # 실제 이미지, 예측 평균 이미지

total_plot_cols = num_cols_per_sample_row # 전체 subplot 그리드의 열 수 (3)
total_plot_rows = num_samples_to_visualize * num_rows_per_sample # 전체 subplot 그리드의 행 수 (샘플 수 * 2)

# Figure 크기 조정
# 각 서브플롯당 대략적인 크기를 5x5 인치로 가정하여 전체 Figure 크기를 계산합니다.
plt.figure(figsize=(total_plot_cols * 5, total_plot_rows * 5)) 

print(f"\n총 {num_samples_to_visualize}개의 샘플을 시각화합니다 (인덱스 {start_sample_idx}부터).")
print(f"생성될 시각화 그리드: {total_plot_rows} 행 x {total_plot_cols} 열")

for i, current_sample_relative_idx in enumerate(range(num_samples_to_visualize)):
    bidx = sequential_batch_idxs[current_sample_relative_idx]
    terrain, fire_seq, tnh_seq, wind_seq = test_list[bidx]

    print(f"\n--- Processing Sample Index: {bidx} ---")

    start_time = time.time()
    mean_preds, variance_preds, real_scores, fake_scores = ensemble_chaining_with_disc_scores(
        gen, disc, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=9
    )
    end_time = time.time()
    prediction_time = end_time - start_time
    print(f"Prediction Time for Sample {bidx}: {prediction_time:.2f} seconds")

    mean_preds_np = mean_preds.numpy()
    target_seq_np = fire_seq[0, 1:4].numpy()
    real_scores_np = np.array([score.numpy() for score in real_scores])
    fake_scores_np = np.array([score.numpy() for score in fake_scores])

    # FID 계산을 위해 현재 배치의 이미지들 저장
    all_gt_images_cgan.extend(target_seq_np) # (3, H, W, C) 이미지가 추가됨
    all_pred_images_cgan.extend(mean_preds_np[0]) # (3, H, W, C) 이미지가 추가됨

    num_steps = mean_preds_np.shape[1]
    hours = [4, 8, 12]

    current_sample_start_row = i * 2

    for t in range(num_steps):
        gt_img_tensor = tf.convert_to_tensor(target_seq_np[t], dtype=tf.float32)
        mean_img_tensor = tf.convert_to_tensor(mean_preds_np[0, t], dtype=tf.float32)

        # MSE 계산
        mse_loss = tf.keras.losses.MeanSquaredError()
        mse_score = mse_loss(gt_img_tensor, mean_img_tensor).numpy().item()

        # SSIM 계산
        ssim_score = tf.image.ssim(
            tf.expand_dims(gt_img_tensor, axis=0),
            tf.expand_dims(mean_img_tensor, axis=0),
            max_val=1.0
        ).numpy().item()

        # IoU 계산 (새로 추가)
        iou_score = calculate_iou(gt_img_tensor, mean_img_tensor, threshold=0.1)

        gt_img = np.squeeze(gt_img_tensor.numpy())
        mean_img = np.squeeze(mean_img_tensor.numpy())

        # 실제 이미지 subplot
        plt.subplot(total_plot_rows, total_plot_cols, current_sample_start_row * total_plot_cols + t + 1)
        plt.imshow(gt_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"True {hours[t]}h (Idx {bidx})\n(Score: {real_scores_np[t]:.3f})")
        plt.axis('off')

        # 예측 이미지 subplot (제목에 IoU 추가)
        plt.subplot(total_plot_rows, total_plot_cols, (current_sample_start_row + 1) * total_plot_cols + t + 1)
        plt.imshow(mean_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"Pred Mean {hours[t]}h (Idx {bidx})\n"
                  f"(D-Score: {fake_scores_np[t]:.3f}, MSE: {mse_score:.4f}\n"
                  f"SSIM: {ssim_score:.4f}, IoU: {iou_score:.4f})") # IoU 추가
        plt.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# --- FID 점수 계산 및 출력 (루프 종료 후) ---
print("\n--- Calculating FID Score for CGAN Model ---")
# 리스트를 텐서로 변환 (N, H, W, C)
all_gt_tensor_cgan = tf.convert_to_tensor(all_gt_images_cgan, dtype=tf.float32)
all_pred_tensor_cgan = tf.convert_to_tensor(all_pred_images_cgan, dtype=tf.float32)

# FID 계산
fid_score_cgan = calculate_fid(inception_model, all_gt_tensor_cgan, all_pred_tensor_cgan)
print(f"FID Score (CGAN vs. Ground Truth): {fid_score_cgan:.4f}")

In [ ]:
import time
import matplotlib.pyplot as plt
def ensemble_chaining_with_disc_scores(generator, discriminator, 
                                     terrain, fire_seq, tnh_seq, wind_seq, 
                                     n_ensemble=9):
    """
    앙상블 체이닝 추론을 수행하며, 각 단계별 판별자 점수를 함께 반환합니다. (Shape 오류 수정 버전)
    """
    # fire_seq: (batch, time_steps, H, W, C) -> fire_seq[:, 0, ...]는 (batch, H, W, C)
    initial_fire_input = fire_seq[:, 0, ...] # 0h (현재) fire state
    
    mean_predictions = []
    variance_predictions = []
    real_scores = []
    fake_scores = []
    
    current_fire_input = initial_fire_input # (batch, H, W, C)
    
    # 4h, 8h, 12h 예측 (총 3 스텝)
    # tnh_seq.shape[1]은 4 (0h, 4h, 8h, 12h) 이므로 range(4)가 될 것.
    # 하지만 예측은 4h, 8h, 12h (3스텝)이므로 t+1을 사용하기 위해 range(3)으로 제한합니다.
    for t in range(3): # 0, 1, 2 에 해당 (각각 4h, 8h, 12h 예측)
        step_indices = tf.one_hot(t, depth=3) # t는 0, 1, 2
        step_indices = tf.expand_dims(step_indices, axis=0) # (1, 3)

        # tnh_seq와 wind_seq에서 해당 시간 스텝 데이터 추출
        # tnh_seq: (batch, 4, 1, 2) -> tnh_seq[:, t+1, ...] 는 (batch, 1, 2)
        # wind_seq: (batch, 4, 2, 1) -> wind_seq[:, t+1, ...] 는 (batch, 2, 1)
        # 예측은 fire_seq의 t+1 스텝 (4h, 8h, 12h)의 날씨 정보를 사용해야 함.
        tnh_t = tnh_seq[:, t]  # 0h 인덱스가 아니라 예측하려는 스텝의 날씨 정보
        wind_t = wind_seq[:, t]
        
        # --- 앙상블 예측 ---
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM)) # 배치 크기 1에 맞춰 (1, LATENT_DIM)
            
            # 제너레이터 입력: [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise]
            # current_fire_input: (1, H, W, C)
            # terrain: (1, H, W, C)
            # tnh_t: (1, 1, 2)
            # wind_t: (1, 2, 1)
            # step_indices: (1, 3)
            # noise: (1, LATENT_DIM)
            
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0) # (n_ensemble, H, W, C)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0) # (H, W, C)
        variance_pred = tf.math.reduce_variance(ensemble_tensor, axis=0) # (H, W, C)

        mean_predictions.append(tf.expand_dims(mean_pred, axis=0)) # (1, H, W, C) -> 각 스텝 예측 저장
        variance_predictions.append(tf.expand_dims(variance_pred, axis=0)) # (1, H, W, C)

        # --- 판별자 점수 계산 ---
        mean_pred_batched = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C) - 판별자 입력을 위해 배치 차원 추가
        
        # true_next_fire: fire_seq에서 예측하려는 다음 실제 불 이미지 (fire_seq[:, t+1, ...])
        true_next_fire = fire_seq[:, t + 1, ...] # (1, H, W, C)
        
        disc_inputs_real = [true_next_fire, terrain, tnh_t, wind_t, step_indices]
        disc_inputs_fake = [mean_pred_batched, terrain, tnh_t, wind_t, step_indices]

        # discriminator는 여러 개의 패치에 대한 점수를 반환할 수 있으므로, 평균을 취함
        real_score_patches = discriminator(disc_inputs_real, training=False) # (1, num_patches) 또는 (1, 1)
        fake_score_patches = discriminator(disc_inputs_fake, training=False) # (1, num_patches) 또는 (1, 1)
        
        avg_real_score = tf.reduce_mean(real_score_patches) # 스칼라
        avg_fake_score = tf.reduce_mean(fake_score_patches) # 스칼라
        
        real_scores.append(avg_real_score)
        fake_scores.append(avg_fake_score)
        
        # 다음 스텝의 입력으로 현재 예측 결과를 사용
        # 현재 mean_pred는 (H, W, C) 이므로, current_fire_input (1, H, W, C)로 만들어야 함
        current_fire_input = tf.expand_dims(mean_pred, axis=0) # (1, H, W, C)

    # 결과를 쌓아서 반환
    # mean_predictions: 각 요소가 (1, H, W, C) -> stacked_means: (3, 1, H, W, C) -> tf.stack(axis=1)하면 (1, 3, H, W, C)
    stacked_means = tf.stack(mean_predictions, axis=1) # (batch, num_steps, H, W, C)
    stacked_vars = tf.stack(variance_predictions, axis=1) # (batch, num_steps, H, W, C)
    
    # real_scores/fake_scores는 각 스텝별 스칼라 텐서 리스트 (3개)
    return stacked_means, stacked_vars, real_scores, fake_scores

# --- 메인 코드 (수정 불필요) ---

# 테스트 데이터셋 불러오기
#test_tfrecord = "./ignition/test_big_128_all.tfrecord"
test_tfrecord = "./ignition/test_128_56ea.tfrecord"
test_dataset = tf.data.TFRecordDataset([test_tfrecord])
test_dataset = test_dataset.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
test_list = list(test_dataset)
num_batches = len(test_list)
print(len(test_list))
# 원하는 샘플 개수 선택
# --- 원하는 샘플 개수와 시작 인덱스 설정 ---
NN = 0 # 0부터 시작, 0이면 0~35, 1이면 36~71 등 36개 단위로 이동
start_sample_idx = 28 * NN # 0부터 시작하여 시각화를 시작할 샘플의 인덱스
num_samples_to_visualize = 28 # 원하는 샘플 수 (예: 1~36)

# 인덱스 유효성 검사 및 조정
if start_sample_idx >= num_batches:
    print(f"시작 인덱스 {start_sample_idx}가 전체 샘플 수 {num_batches}보다 크거나 같습니다. 시각화할 샘플이 없습니다.")
    exit()

if start_sample_idx + num_samples_to_visualize > num_batches:
    num_samples_to_visualize = num_batches - start_sample_idx
    print(f"경고: 요청된 샘플 수가 전체 데이터셋을 초과합니다. {num_samples_to_visualize}개의 샘플만 시각화합니다.")

if num_samples_to_visualize <= 0:
    print("시각화할 샘플이 없습니다. 시작 인덱스와 샘플 수를 확인하세요.")
    exit()

# 순차적인 인덱스 리스트 생성
sequential_batch_idxs = list(range(start_sample_idx, start_sample_idx + num_samples_to_visualize))

# --- 시각화 설정 ---
# 각 샘플은 2행 (실제, 예측) x 3열 (4h, 8h, 12h)을 차지합니다.
num_cols_per_sample_row = 3 # 4h, 8h, 12h
num_rows_per_sample = 2 # 실제 이미지, 예측 평균 이미지

total_plot_cols = num_cols_per_sample_row # 전체 subplot 그리드의 열 수 (3)
total_plot_rows = num_samples_to_visualize * num_rows_per_sample # 전체 subplot 그리드의 행 수 (샘플 수 * 2)

# Figure 크기 조정
# 각 서브플롯당 대략적인 크기를 5x5 인치로 가정하여 전체 Figure 크기를 계산합니다.
plt.figure(figsize=(total_plot_cols * 5, total_plot_rows * 5)) 

print(f"\n총 {num_samples_to_visualize}개의 샘플을 시각화합니다 (인덱스 {start_sample_idx}부터).")
print(f"생성될 시각화 그리드: {total_plot_rows} 행 x {total_plot_cols} 열")

for i, current_sample_relative_idx in enumerate(range(num_samples_to_visualize)):
    bidx = sequential_batch_idxs[current_sample_relative_idx] 
    terrain, fire_seq, tnh_seq, wind_seq = test_list[bidx]
    
    print(f"\n--- Processing Sample Index: {bidx} ---")
    
    # 예측 시간 측정을 위해 시작 시간 기록
    start_time = time.time()
    
    # ensemble_chaining_with_disc_scores 함수 호출
    mean_preds, variance_preds, real_scores, fake_scores = ensemble_chaining_with_disc_scores(
        gen, disc, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=9
    )
    
    # 예측 종료 시간 기록 및 소요 시간 계산
    end_time = time.time()
    prediction_time = end_time - start_time
    print(f"Prediction Time for Sample {bidx}: {prediction_time:.2f} seconds")

    # Numpy 배열로 변환
    mean_preds_np = mean_preds.numpy()
    target_seq_np = fire_seq[0, 1:4].numpy()
    real_scores_np = np.array([score.numpy() for score in real_scores])
    fake_scores_np = np.array([score.numpy() for score in fake_scores])
    
    num_steps = mean_preds_np.shape[1]
    hours = [4, 8, 12]

    # 각 샘플에 대한 서브플롯 위치 계산
    current_sample_start_row = i * num_rows_per_sample 
    
    # MSE와 SSIM 점수를 저장할 리스트
    mse_scores = []
    ssim_scores = []

    for t in range(num_steps):
        gt_img_tensor = tf.convert_to_tensor(target_seq_np[t], dtype=tf.float32)
        mean_img_tensor = tf.convert_to_tensor(mean_preds_np[0, t], dtype=tf.float32)

        # MSE 계산
        mse_loss = tf.keras.losses.MeanSquaredError()
        mse_score = mse_loss(gt_img_tensor, mean_img_tensor).numpy().item()
        mse_scores.append(mse_score)

        # SSIM 계산 (이미지는 채널 차원이 필요하므로 expand_dims 사용)
        ssim_score = tf.image.ssim(
            tf.expand_dims(gt_img_tensor, axis=0), 
            tf.expand_dims(mean_img_tensor, axis=0), 
            max_val=1.0
        ).numpy().item()
        ssim_scores.append(ssim_score)
        
        gt_img = np.squeeze(gt_img_tensor.numpy())
        mean_img = np.squeeze(mean_img_tensor.numpy())
        
        # 실제 이미지 subplot
        plt.subplot(total_plot_rows, total_plot_cols, current_sample_start_row * total_plot_cols + t + 1)
        plt.imshow(gt_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"True {hours[t]}h (Idx {bidx})\n(Score: {real_scores_np[t]:.3f})")
        plt.axis('off')

        # 예측 이미지 subplot (제목에 점수 추가)
        plt.subplot(total_plot_rows, total_plot_cols, (current_sample_start_row + 1) * total_plot_cols + t + 1)
        plt.imshow(mean_img, cmap='hot', vmin=0, vmax=1)
        plt.title(f"Pred Mean {hours[t]}h (Idx {bidx})\n"
                  f"(D-Score: {fake_scores_np[t]:.3f}, MSE: {mse_scores[t]:.4f}, SSIM: {ssim_scores[t]:.4f})")
        plt.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
from tqdm import tqdm

# --- 2. 평가를 위한 예측 및 점수 계산 함수 ---

def get_ensemble_predictions(generator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=9):
    """
    하나의 샘플에 대해 자기회귀 및 앙상블 예측을 수행합니다.
    (Discriminator 점수 계산이 없는 평가용 경량 버전)
    """
    initial_fire_input = fire_seq[:, 0, ...]
    mean_predictions = []
    current_fire_input = initial_fire_input
    
    for t in range(3): # 4h, 8h, 12h 예측
        step_indices = tf.one_hot(t, depth=3)
        step_indices = tf.expand_dims(step_indices, axis=0)
        tnh_t = tnh_seq[:, t]
        wind_t = wind_seq[:, t]
        
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM))
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0)
        mean_predictions.append(tf.expand_dims(mean_pred, axis=0))
        current_fire_input = tf.expand_dims(mean_pred, axis=0)

    return tf.stack(mean_predictions, axis=1) # (batch, num_steps, H, W, C)

def evaluate_model(generator, dataset, n_ensemble=9):
    """데이터셋 전체에 대한 평균 MSE와 SSIM 점수를 계산합니다."""
    all_mse_scores = [[] for _ in range(3)] # [4h_scores, 8h_scores, 12h_scores]
    all_ssim_scores = [[] for _ in range(3)]
    all_iou_scores =  [[] for _ in range(3)]
    mse_loss_fn = tf.keras.losses.MeanSquaredError()

    for batch in tqdm(dataset, desc="Evaluating Model"):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        
        # 모델 예측
        mean_preds = get_ensemble_predictions(generator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble)
        
        # 정답 시퀀스
        target_seq = fire_seq[:, 1:4, ...] # 4h, 8h, 12h

        for t in range(3): # 각 시간 단계별로 점수 계산
            pred_t = mean_preds[:, t, ...]
            target_t = target_seq[:, t, ...]

            # MSE 계산
            mse = mse_loss_fn(target_t, pred_t).numpy()
            all_mse_scores[t].append(mse)

            # SSIM 계산
            ssim = tf.image.ssim(target_t, pred_t, max_val=1.0).numpy().mean()
            all_ssim_scores[t].append(ssim)
            
            intersection = tf.reduce_sum(pred_t * target_t)
            union = tf.reduce_sum(pred_t) + tf.reduce_sum(target_t) - intersection
            
            iou = (intersection + 1e-7) / (union + 1e-7)
            all_iou_scores[t].append(iou.numpy())
            
    # 각 시간대별 평균 점수 계산
    avg_mse = [np.mean(scores) for scores in all_mse_scores]
    avg_ssim = [np.mean(scores) for scores in all_ssim_scores]
    avg_iou = [np.mean(scores) for scores in all_iou_scores]
    return {"MSE": avg_mse, "SSIM": avg_ssim, "IoU": avg_iou}

# --- 3. 메인 실행 및 시각화 ---

# 학습된 Generator 모델 로드 (예시)
# gen = tf.keras.models.load_model('path/to/your/generator_model')

# 데이터셋 로드
print("Loading datasets...")
train_tfrecord = "./ignition/train_mid_128_all.tfrecord"
train_dataset = tf.data.TFRecordDataset([train_tfrecord])
train_dataset = train_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
train_dataset = train_dataset.take(500)


test_tfrecord = "./ignition/test_mid_128_all.tfrecord"
test_dataset = tf.data.TFRecordDataset([test_tfrecord])
test_dataset = test_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.take(500)
print("Datasets loaded.")

# 모델 평가 실행
print("\nEvaluating on Test Dataset...")
test_results = evaluate_model(gen, test_dataset)
print(f"Test Dataset  -> MSE: {test_results['MSE']}, SSIM: {test_results['SSIM']}")
print("\nEvaluating on Train Dataset...")
train_results = evaluate_model(gen, train_dataset)
print(f"Train Dataset -> MSE: {train_results['MSE']}, SSIM: {train_results['SSIM']}")
# 결과 출력
print("\n--- Evaluation Results ---")




# --- 시각화 ---

def plot_results(train_res, test_res, model_name=""):
    labels = ['4h', '8h', '12h']
    x = np.arange(len(labels))
    width = 0.35

    # subplots를 1x2에서 1x3으로 변경하고, figsize 너비를 늘립니다.
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 6))
    fig.suptitle(f'{model_name} Model Evaluation', fontsize=16)

    # MSE 그래프 (기존과 동일)
    rects1 = ax1.bar(x - width/2, train_res['MSE'], width, label='Train')
    rects2 = ax1.bar(x + width/2, test_res['MSE'], width, label='Test')
    ax1.set_ylabel('Mean Squared Error (MSE)')
    ax1.set_title('Average MSE (Lower is Better)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.bar_label(rects1, padding=3, fmt='%.4f')
    ax1.bar_label(rects2, padding=3, fmt='%.4f')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # SSIM 그래프 (기존과 동일)
    rects3 = ax2.bar(x - width/2, train_res['SSIM'], width, label='Train')
    rects4 = ax2.bar(x + width/2, test_res['SSIM'], width, label='Test')
    ax2.set_ylabel('Structural Similarity (SSIM)')
    ax2.set_title('Average SSIM (Higher is Better)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.legend()
    ax2.bar_label(rects3, padding=3, fmt='%.4f')
    ax2.bar_label(rects4, padding=3, fmt='%.4f')
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    
    # --- IoU 그래프 추가 ---
    rects5 = ax3.bar(x - width/2, train_res['IoU'], width, label='Train')
    rects6 = ax3.bar(x + width/2, test_res['IoU'], width, label='Test')
    ax3.set_ylabel('Intersection over Union (IoU)')
    ax3.set_title('Average IoU (Higher is Better)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels)
    ax3.legend()
    ax3.bar_label(rects5, padding=3, fmt='%.4f')
    ax3.bar_label(rects6, padding=3, fmt='%.4f')
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    # --------------------
    
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print("\nPlotting results...")
plot_results(train_results, test_results)


In [ ]:
print(test_results)

In [ ]:
# 데이터셋 로드
print("Loading datasets...")
train_tfrecord = "./ignition/train_big_128_all.tfrecord"
train_dataset = tf.data.TFRecordDataset([train_tfrecord])
train_dataset = train_dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
train_dataset = train_dataset.take(400)

print("\nEvaluating on Train Dataset...")
train_results = evaluate_model(gen, train_dataset)

# 결과 출력
print("\n--- Evaluation Results ---")
print(f"Train Dataset -> MSE: {train_results['MSE']}, SSIM: {train_results['SSIM']}")
print(f"Test Dataset  -> MSE: {test_results['MSE']}, SSIM: {test_results['SSIM']}")


# --- 시각화 ---

def plot_results(train_res, test_res, model_name=""):
    labels = ['4h', '8h', '12h']
    x = np.arange(len(labels))
    width = 0.35

    # subplots를 1x2에서 1x3으로 변경하고, figsize 너비를 늘립니다.
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 6))
    fig.suptitle(f'{model_name} Model Evaluation', fontsize=16)

    # MSE 그래프 (기존과 동일)
    rects1 = ax1.bar(x - width/2, train_res['MSE'], width, label='Train')
    rects2 = ax1.bar(x + width/2, test_res['MSE'], width, label='Test')
    ax1.set_ylabel('Mean Squared Error (MSE)')
    ax1.set_title('Average MSE (Lower is Better)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.bar_label(rects1, padding=3, fmt='%.4f')
    ax1.bar_label(rects2, padding=3, fmt='%.4f')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # SSIM 그래프 (기존과 동일)
    rects3 = ax2.bar(x - width/2, train_res['SSIM'], width, label='Train')
    rects4 = ax2.bar(x + width/2, test_res['SSIM'], width, label='Test')
    ax2.set_ylabel('Structural Similarity (SSIM)')
    ax2.set_title('Average SSIM (Higher is Better)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.legend()
    ax2.bar_label(rects3, padding=3, fmt='%.4f')
    ax2.bar_label(rects4, padding=3, fmt='%.4f')
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    
    # --- IoU 그래프 추가 ---
    rects5 = ax3.bar(x - width/2, train_res['IoU'], width, label='Train')
    rects6 = ax3.bar(x + width/2, test_res['IoU'], width, label='Test')
    ax3.set_ylabel('Intersection over Union (IoU)')
    ax3.set_title('Average IoU (Higher is Better)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels)
    ax3.legend()
    ax3.bar_label(rects5, padding=3, fmt='%.4f')
    ax3.bar_label(rects6, padding=3, fmt='%.4f')
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    # --------------------
    
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print("\nPlotting results...")
plot_results(train_results, test_results)

In [ ]:
print("A random sample has been selected from the training dataset.")

import matplotlib.gridspec as gridspec
# --- 2. 단일 스텝 앙상블 예측 함수 ---
BUFFER_SIZE = 1000 

# 데이터셋을 섞고(shuffle), 그 중 하나만(take(1)) 가져와 새로운 데이터셋을 만듭니다.
random_sample_dataset = train_dataset.shuffle(buffer_size=BUFFER_SIZE).take(1)

# 위에서 만든 '1개짜리 랜덤 데이터셋'에서 샘플을 가져옵니다.
sample_batch = next(iter(random_sample_dataset))
terrain, fire_seq, tnh_seq, wind_seq = sample_batch
# --- 2. 12시간째 앙상블 예측을 위한 함수 (새로 작성) ---

def generate_12h_ensemble_visuals(generator, terrain, fire_seq, tnh_seq, wind_seq, n_ensemble=9):
    """
    자기회귀 체인을 거쳐 12시간째의 앙상블 예측을 수행하고,
    개별 예측 결과들과 평균 예측 결과를 반환합니다.
    """
    current_fire_input = fire_seq[:, 0, ...] # 0h (현재) fire state
    
    # 4h, 8h 예측을 순차적으로 수행 (다음 단계의 입력으로 사용하기 위함)
    print("Performing autoregressive prediction for 4h and 8h...")
    for t in range(2): # 0(->4h), 1(->8h) 예측
        step_indices = tf.one_hot(t, depth=3)
        step_indices = tf.expand_dims(step_indices, axis=0)
        tnh_t = tnh_seq[:, t]
        wind_t = wind_seq[:, t]
        
        # 중간 과정은 앙상블의 평균만 사용
        step_ensemble_preds = []
        for _ in range(n_ensemble):
            noise = tf.random.normal(shape=(1, LATENT_DIM))
            pred = generator(
                [current_fire_input, terrain, tnh_t, wind_t, step_indices, noise],
                training=False
            )
            step_ensemble_preds.append(pred)

        ensemble_tensor = tf.concat(step_ensemble_preds, axis=0)
        mean_pred = tf.reduce_mean(ensemble_tensor, axis=0)
        current_fire_input = tf.expand_dims(mean_pred, axis=0)

    # 이제 current_fire_input은 8h 예측 결과가 됨
    
    # --- 최종 12h 앙상블 예측 (시각화 대상) ---
    print(f"Generating {n_ensemble} ensemble members for 12h prediction...")
    t_final = 2 # 8h -> 12h 예측을 위한 인덱스
    step_indices_final = tf.one_hot(t_final, depth=3)
    step_indices_final = tf.expand_dims(step_indices_final, axis=0)
    tnh_final = tnh_seq[:, t_final]
    wind_final = wind_seq[:, t_final]
    
    ensemble_preds_12h = []
    for _ in range(n_ensemble):
        noise = tf.random.normal(shape=(1, LATENT_DIM))
        pred_12h = generator(
            [current_fire_input, terrain, tnh_final, wind_final, step_indices_final, noise],
            training=False
        )
        ensemble_preds_12h.append(tf.squeeze(pred_12h, axis=0))

    ensemble_tensor_12h = tf.stack(ensemble_preds_12h, axis=0)
    mean_pred_12h = tf.reduce_mean(ensemble_tensor_12h, axis=0)

    return ensemble_preds_12h, mean_pred_12h

# --- 3. 시각화 실행 (수정 없음) ---

# 학습된 Generator 모델 로드 (예시)
# gen = tf.keras.models.load_model('path/to/your/generator_model')

# 12시간째 앙상블 예측 수행
individual_preds_12h, mean_pred_12h = generate_12h_ensemble_visuals(
    gen, terrain, fire_seq, tnh_seq, wind_seq
)

print("Visualization starting...")

# (이전 답변과 동일한 시각화 코드)
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 4, figure=fig)
fig.suptitle('Ensemble Sampling for 12h Prediction (after Autoregression)', fontsize=16)

# 왼쪽 3x3 그리드에 9개 앙상블 멤버 시각화
for i in range(9):
    ax = fig.add_subplot(gs[i//3, i%3])
    ax.imshow(individual_preds_12h[i], cmap='hot', vmin=0, vmax=1)
    ax.set_title(f'12h Ensemble Member {i+1}')
    ax.axis('off')

# 오른쪽 큰 셀에 앙상블 평균 시각화
ax_mean = fig.add_subplot(gs[:, 3])
ax_mean.imshow(mean_pred_12h, cmap='hot', vmin=0, vmax=1)
ax_mean.set_title('12h Ensemble Average', fontsize=14)
ax_mean.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
train_small = load_subdataset("./train_mid_tanh.tfrecord", batch_size=8)
fixed_sample = next(iter(train_small))# 랜덤 샘플 선택
# Unpack the sample
(spread_0h, spread_4h, spread_8h, spread_12h, 
 terrain, dem,fuel, t_h_t0, t_h_t1, t_h_t2, 
 ws_wd_t0, ws_wd_t1, ws_wd_t2, 
 ignite_index, weather_index) = fixed_sample

# Helper function to visualize grayscale images
def show_grayscale(image, title, ax):
    ax.imshow(tf.squeeze(image), cmap='gray')
    ax.axis('off')
    ax.set_title(title)

# Helper function to visualize RGB images
def show_rgb(image, title, ax):
    ax.imshow(image)
    ax.axis('off')
    ax.set_title(title)

# Helper function to visualize heatmaps
def show_heatmap(data, title, ax):
    ax.imshow(tf.squeeze(data), cmap='viridis')
    ax.axis('off')
    ax.set_title(title)

# Plotting
fig, axes = plt.subplots(3, 4, figsize=(12, 9))

# Spread images
show_grayscale(spread_0h[0], 'Spread 0h', axes[0, 0])
show_grayscale(spread_4h[0], 'Spread 4h', axes[0, 1])
show_grayscale(spread_8h[0], 'Spread 8h', axes[0, 2])
show_grayscale(spread_12h[0], 'Spread 12h', axes[0, 3])

# Terrain image
show_rgb(dem[0]/2+0.5, 'Dem', axes[1, 0])
show_rgb(fuel[0]/2+0.5, 'Fuel', axes[1, 1])

# Temperature & Humidity
show_heatmap(t_h_t0[0], 'Temp/Hum 0h', axes[1, 2])
show_heatmap(t_h_t1[0], 'Temp/Hum 4h', axes[1, 3])
show_heatmap(t_h_t2[0], 'Temp/Hum 8h', axes[2, 0])

# Wind speed & direction
show_heatmap(ws_wd_t0[0], 'Wind 0h', axes[2, 1])
show_heatmap(ws_wd_t1[0], 'Wind 4h', axes[2, 2])
show_heatmap(ws_wd_t2[0], 'Wind 8h', axes[2, 3])

# Ignite & Weather index
axes[2, 3].text(0.5, 0.5, f'Ignite: {ignite_index[0].numpy()}\nWeather: {weather_index[0].numpy()}',
                 fontsize=12, ha='center', va='center')
axes[2, 3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
cond_gan.fit(train_small, epochs=200)

In [ ]:
test_small = load_subdataset("./test_small_tanh.tfrecord", batch_size=8)

# 확인할 샘플 수 지정 (예: 5개)
num_samples_to_check = 5

# test dataset에서 지정한 배치 수만큼 가져옵니다.
for i, sample in enumerate(test_small.take(num_samples_to_check)):
    # sample의 unpack: 순서대로
    # spread_0h, spread_4h, spread_8h, spread_12h,
    # terrain, dem, fuel, t_h_t0, t_h_t1, t_h_t2,
    # ws_wd_t0, ws_wd_t1, ws_wd_t2,
    # ignite_index, weather_index
    (spread_0h, spread_4h, spread_8h, spread_12h, 
     terrain, dem, fuel, 
     t_h_t0, t_h_t1, t_h_t2, 
     ws_wd_t0, ws_wd_t1, ws_wd_t2, 
     ignite_index, weather_index) = sample

    # 모델의 입력 구성 (배치 내 첫 샘플만 사용)
    model_inputs = [
        terrain[0:1],          # "terrain" : (256,256,3)
        spread_0h[0:1],        # "initial_fire" : (256,256,1)
        t_h_t0[0:1],           # "tnh_0" : (4,1,2)
        t_h_t1[0:1],           # "tnh_1" : (4,1,2)
        t_h_t2[0:1],           # "tnh_2" : (4,1,2)
        ws_wd_t0[0:1],         # "wind_0" : (4,1,2)
        ws_wd_t1[0:1],         # "wind_1" : (4,1,2)
        ws_wd_t2[0:1]          # "wind_2" : (4,1,2)
    ]
    
    # cond_gan의 generator를 사용하여 예측 (출력 shape: (1, 3, 256, 256, 1))
    pred_seq = cond_gan.generator.predict(model_inputs)
    
    # 예측 결과: 각 타임스텝은 순서대로 [0]=예측 4h, [1]=예측 8h, [2]=예측 12h
    pred_4h = pred_seq[0, 0, :, :, 0]
    pred_8h = pred_seq[0, 1, :, :, 0]
    pred_12h = pred_seq[0, 2, :, :, 0]
    
    # ground truth: 각 time step의 실제 spread 이미지 (배치 내 첫 샘플 선택)
    gt_4h = spread_4h[0].numpy().squeeze()   # shape (256,256)
    gt_8h = spread_8h[0].numpy().squeeze()
    gt_12h = spread_12h[0].numpy().squeeze()
    
    # 시각화: 예측 결과와 실제 정답을 비교
    fig, axes = plt.subplots(3, 2, figsize=(10, 15))
    
    # 4시간 후 예측 vs 실제
    axes[0, 0].imshow(pred_4h, cmap='hot')
    axes[0, 0].set_title("Predicted Spread 4h")
    axes[0, 0].axis('off')
    axes[0, 1].imshow(gt_4h, cmap='hot')
    axes[0, 1].set_title("Ground Truth Spread 4h")
    axes[0, 1].axis('off')
    
    # 8시간 후 예측 vs 실제
    axes[1, 0].imshow(pred_8h, cmap='hot')
    axes[1, 0].set_title("Predicted Spread 8h")
    axes[1, 0].axis('off')
    axes[1, 1].imshow(gt_8h, cmap='hot')
    axes[1, 1].set_title("Ground Truth Spread 8h")
    axes[1, 1].axis('off')
    
    # 12시간 후 예측 vs 실제
    axes[2, 0].imshow(pred_12h, cmap='hot')
    axes[2, 0].set_title("Predicted Spread 12h")
    axes[2, 0].axis('off')
    axes[2, 1].imshow(gt_12h, cmap='hot')
    axes[2, 1].set_title("Ground Truth Spread 12h")
    axes[2, 1].axis('off')
    
    plt.suptitle(f"Test Batch Sample {i+1}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:

print("terrain shape:", terrain.shape)         # (1,256,256,3)
print("weather_seq shape:", weather_seq.shape)   # (1,3,256,256,4)
print("spread_seq shape:", spread_seq.shape)  
fixed_sample = list(train_ds.take(1))[0]
fixed_terrain = tf.expand_dims(terrain[0], axis=0)        # shape: (1,256,256,3)
fixed_weather = tf.expand_dims(weather_seq[0], axis=0) 
fixed_inputs = (fixed_terrain,fixed_weather )
print(fixed_inputs[0].shape)


In [ ]:
# 데이터셋에서 하나의 샘플을 추출하여 시각화합니다.
for terrain, weather_seq, spread_seq in fixed_sample:
    # terrain: (1,256,256,3)
    # weather_seq: (1,3,256,256,4)
    # spread_seq: (1,3,256,256,1)
    terrain_np = terrain[0].numpy()  # (256,256,3)
    weather_np = weather_seq[0].numpy()  # (3,256,256,4)
    spread_np = spread_seq[0].numpy()  # (3,256,256,1)

    # 시각화를 위한 Figure 생성
    plt.figure(figsize=(16, 8))
    
    # 1. Terrain 이미지 (RGB)
    plt.subplot(2, 3, 1)
    plt.imshow(terrain_np)
    plt.title("Terrain (RGB)")
    plt.axis("off")
    
    # 2. Weather 정보: 각 시점별 텍스트로 표시 (온도, 습도, 풍속, 풍향)
    time_labels = ["0h", "4h", "8h"]
    for t in range(3):
        plt.subplot(2, 3, t+2)
        # 기상 데이터는 전체가 동일 값이므로 (256,256,4) 중 좌상단 픽셀만 사용
        weather_vals = weather_np[t, 0, 0, :]  # shape: (4,)
        text_str = (f"Temp: {weather_vals[0]:.2f}\n"
                    f"Hum: {weather_vals[1]:.2f}\n"
                    f"Ws: {weather_vals[2]:.2f}\n"
                    f"Wd: {weather_vals[3]:.2f}")
        plt.text(0.5, 0.5, text_str, fontsize=12, ha='center', va='center', transform=plt.gca().transAxes)
        plt.title(f"Weather {time_labels[t]}")
        plt.axis("off")
    
        plt.tight_layout()
        plt.show()
    
    # 3. Spread 데이터: 각 시점별 산불 확산 이미지 (그레이스케일)
    plt.figure(figsize=(12, 4))
    spread_time_labels = ["4h", "8h", "12h"]
    for t in range(3):
        plt.subplot(1, 3, t+1)
        # spread_np[t]: (256,256,1) → squeeze해서 (256,256)
        plt.imshow(np.squeeze(spread_np[t]), cmap='gray')
        plt.title(f"Spread {spread_time_labels[t]}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
class SavePredictions(tf.keras.callbacks.Callback):
    """
    매 10 에포크마다 고정 샘플(terrain, weather_seq)에 대해 Generator의 예측 결과를 저장하는 콜백.
    출력 이미지(각 시간대: 4h,8h,12h)는 Generator 출력이 tanh를 사용해 [-1,1] 범위일 경우,
    [0,255] 범위로 재조정하여 저장합니다.
    """
    def __init__(self, fixed_sample, output_dir="./predictions"):
        super(SavePredictions, self).__init__()
        self.fixed_sample = fixed_sample  # (terrain, weather_seq)
        self.input = fixed_sample[0],fixed_sample[1],fixed_sample[2]
        self.output_dir = output_dir
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)

    def on_epoch_end(self, epoch, logs=None):
        # 매 10 에포크마다 저장 (에포크 번호는 1부터 시작하도록)
        if (epoch + 1) % 10 == 0:
            # Generator 예측: shape = (batch, 3, 256,256,1)
            predictions = self.model.generator.predict(self.input)
            # 여기서는 batch size가 1이라고 가정합니다.
            pred = predictions[0]  # shape: (3,256,256,1)
            time_labels = ["6h", "12h"]
            for t, label in enumerate(time_labels):
                # 예측 결과에서 채널 차원을 제거: (256,256)
                img = pred[t, ..., 0]
                # tanh 사용 시 [-1,1] 범위를 [0,255]로 변환
                # PIL을 사용하여 그레이스케일 이미지로 변환 및 저장
                im = Image.fromarray(img, mode='L')
                save_path = os.path.join(self.output_dir, f"epoch_{epoch+1}_{label}.png")
                im.save(save_path)
                print(f"Saved prediction: {save_path}")
                
class EpochTracker(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.model.current_epoch = epoch  # 현재 Epoch 정보를 ConditionalGAN에 전달
        self.model.total_epochs = self.params['epochs']  # 전체 Epoch 개수 저장

def parse_tfrecord(example_proto):
    # TFRecord에 저장된 feature 구성
    feature_description = {
        'spread_6h_image': tf.io.FixedLenFeature([], tf.string),
        'spread_12h_image': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'temp_0h': tf.io.FixedLenFeature([], tf.string),
        'hume_0h': tf.io.FixedLenFeature([], tf.string),
        'ws_0h': tf.io.FixedLenFeature([], tf.string),
        'wd_0h': tf.io.FixedLenFeature([], tf.string),
        'temp_6h': tf.io.FixedLenFeature([], tf.string),
        'hume_6h': tf.io.FixedLenFeature([], tf.string),
        'ws_6h': tf.io.FixedLenFeature([], tf.string),
        'wd_6h': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64),
    }
    features = tf.io.parse_single_example(example_proto, feature_description)

    # [1] Spread images (Grayscale, (256,256,1))
    def decode_and_reshape_spread(key):
        # 저장 시 float32로 정규화된 raw bytes가 저장되어 있다고 가정
        img = tf.io.decode_raw(features[key], tf.float32)
        return tf.reshape(img, [256, 256, 1])
    
    #spread_4h = decode_and_reshape_spread('spread_4h_image')
    #spread_8h = decode_and_reshape_spread('spread_8h_image')
    spread_6h = decode_and_reshape_spread('spread_6h_image')
    spread_12h = decode_and_reshape_spread('spread_12h_image')
    spread_seq = tf.stack([spread_6h, spread_12h], axis=0)  # (3,256,256,1)

    # [2] Terrain image (RGB, (256,256,3))
    terrain = tf.io.decode_raw(features['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [256, 256, 3])
    
    # [3] Weather images: 각 단일 채널 (256,256,1)을 읽은 후 4개 채널씩 합쳐 (256,256,4)로 만들기
    def decode_and_reshape_weather(key):
        img = tf.io.decode_raw(features[key], tf.float32)
        return tf.reshape(img, [256, 256, 1])
    
    temp_0h = decode_and_reshape_weather('temp_0h')
    hume_0h = decode_and_reshape_weather('hume_0h')
    ws_0h   = decode_and_reshape_weather('ws_0h')
    wd_0h   = decode_and_reshape_weather('wd_0h')
    weather_0h = tf.concat([temp_0h, hume_0h, ws_0h, wd_0h], axis=-1)  # (256,256,4)

    temp_6h = decode_and_reshape_weather('temp_6h')
    hume_6h = decode_and_reshape_weather('hume_6h')
    ws_6h   = decode_and_reshape_weather('ws_6h')
    wd_6h   = decode_and_reshape_weather('wd_6h')
    weather_6h = tf.concat([temp_6h, hume_6h, ws_6h, wd_6h], axis=-1)  # (256,256,4)

    #temp_4h = decode_and_reshape_weather('temp_4h')
    #hume_4h = decode_and_reshape_weather('hume_4h')
    #ws_4h   = decode_and_reshape_weather('ws_4h')
    #wd_4h   = decode_and_reshape_weather('wd_4h')
    #weather_4h = tf.concat([temp_4h, hume_4h, ws_4h, wd_4h], axis=-1)  # (256,256,4)

    #temp_8h = decode_and_reshape_weather('temp_8h')
    #hume_8h = decode_and_reshape_weather('hume_8h')
    #ws_8h   = decode_and_reshape_weather('ws_8h')
    #wd_8h   = decode_and_reshape_weather('wd_8h')
    #weather_8h = tf.concat([temp_8h, hume_8h, ws_8h, wd_8h], axis=-1)  # (256,256,4)

    # Stack weather images to form a sequence: (3,256,256,4)
    weather_seq = tf.stack([weather_0h, weather_6h], axis=0)

    # 추가적으로 인덱스도 추출할 수 있음
    ignite_index = features['ignite_index']
    weather_index = features['weather_index']
    
    # 최종적으로 모델 학습에 사용할 형태:  
    # Generator 입력: (terrain, weather_seq)  
    # Target (spread): spread_seq  
    # (추후 Discriminator에도 동일하게 사용)
    return (terrain, weather_seq, spread_seq)


def load_subdataset(tfrecord_path, batch_size=8):
    # TFRecordDataset 생성 및 파싱
    raw_dataset = tf.data.TFRecordDataset(tfrecord_path)
    parsed_dataset = raw_dataset.map(parse_tfrecord, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    
    # 데이터를 충분히 섞어줍니다.
    dataset = parsed_dataset.shuffle(2048, reshuffle_each_iteration=False)
    
    # 각 데이터셋을 batch와 prefetch 처리
    dataset = dataset.batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    
    
    return dataset



In [ ]:
train_small = load_subdataset("./ignition/train_small_2step_tanh.tfrecord", batch_size=32)
fixed_sample = list(train_small.take(1))[0]
noise_input = np.zeros((1,2,256,256,1))
terrain, weather_seq, spread_seq = fixed_sample
fixed_spread = tf.expand_dims(spread_seq[0],axis=0)
fixed_terrain = tf.expand_dims(terrain[0], axis=0)        # shape: (1,256,256,3)
fixed_weather = tf.expand_dims(weather_seq[0], axis=0) 
fixed_inputs = (noise_input, fixed_terrain, fixed_weather, fixed_spread )


# terrain: (1,256,256,3)
# weather_seq: (1,3,256,256,4)
# spread_seq: (1,3,256,256,1)
terrain_np = terrain[0].numpy()  # (256,256,3)
weather_np = weather_seq[0].numpy()  # (3,256,256,4)
spread_np = spread_seq[0].numpy()  # (3,256,256,1)

# 시각화를 위한 Figure 생성
plt.figure(figsize=(16, 8))

# 1. Terrain 이미지 (RGB)
plt.subplot(2, 3, 1)
plt.imshow(terrain_np)
plt.title("Terrain (RGB)")
plt.axis("off")

# 2. Weather 정보: 각 시점별 텍스트로 표시 (온도, 습도, 풍속, 풍향)
time_labels = ["6h", "12h"]
for t in range(2):
    plt.subplot(2, 3, t+2)
    # 기상 데이터는 전체가 동일 값이므로 (256,256,4) 중 좌상단 픽셀만 사용
    weather_vals = weather_np[t, 0, 0, :]  # shape: (4,)
    text_str = (f"Temp: {weather_vals[0]:.2f}\n"
                f"Hum: {weather_vals[1]:.2f}\n"
                f"Ws: {weather_vals[2]:.2f}\n"
                f"Wd: {weather_vals[3]:.2f}")
    plt.text(0.5, 0.5, text_str, fontsize=12, ha='center', va='center', transform=plt.gca().transAxes)
    plt.title(f"Weather {time_labels[t]}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# 3. Spread 데이터: 각 시점별 산불 확산 이미지 (그레이스케일)
plt.figure(figsize=(12, 4))
spread_time_labels = [ "6h", "12h"]
for t in range(2):
    plt.subplot(1, 3, t+1)
    # spread_np[t]: (256,256,1) → squeeze해서 (256,256)
    plt.imshow(np.squeeze(spread_np[t]), cmap='gray')
    plt.title(f"Spread {spread_time_labels[t]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
all_dataset = load_subdataset("./ignition/ds_300_70_256_tanh_2step_gan.tfrecord", batch_size=32)


In [ ]:

all_train = all_dataset.take(315)
all_test = all_dataset.skip(315)


In [ ]:
"""
train model with small_subset train data
"""
save_pred_cb = SavePredictions(fixed_sample=fixed_inputs, output_dir="./ignition/lstm_check")
epoch_tracker = EpochTracker()
cond_gan.fit(train_small, epochs=300, callbacks=[epoch_tracker, save_pred_cb])

In [ ]:
"""
show test dataset prediction image
"""
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import time
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim
import tensorflow as tf

# 테스트 데이터셋에서 샘플 수 설정
num_samples = 10
test_small = load_subdataset("./ignition/test_small_2step_tanh.tfrecord", batch_size=16)
#test_small = load_subdataset("test_small.tfrecord", batch_size=16)
# test_dataset에서 데이터를 추출하기 위한 반복자 생성
test_iterator = iter(test_small)

# 생성된 이미지 저장
start_time = time.time()
generated_images = []
real_images = []

total_samples_collected = 0  # 수집된 샘플 수 추적

while total_samples_collected < num_samples:
    try:
        # test_dataset에서 하나의 배치를 가져옴
        sample_batch = next(test_iterator)
        terrain, weather_seq, spread_seq = sample_batch
    # terrain: (1,256,256,3)
    # weather_seq: (1,3,256,256,4)
    # spread_seq: (1,3,256,256,1)
        


        # 배치 내의 각 샘플을 순회하여 처리
        for i in range(terrain.shape[0]):
            if total_samples_collected >= num_samples:
                break

            # noise 생성
            noise_input = tf.random.normal(shape=(1,2,256,256,1), mean=0.0, stddev=0.2)
            terrain_np = terrain[i:i+1].numpy()  # (256,256,3)
            weather_np = weather_seq[i:i+1].numpy()  # (2,256,256,4)
            spread_np = spread_seq[i:i+1].numpy()  # (2,256,256,1)
            fixed_terrain = tf.expand_dims(terrain_np, axis=0)        # shape: (1,256,256,3)
            fixed_weather = tf.expand_dims(weather_np, axis=0) 
            # 하나의 샘플만 선택

            # generator로 이미지 생성
            generated_image = generator.predict([noise_input, terrain_np, weather_np])
            print(generated_image.shape)
            generated_image = generated_image[0]
            generated_image = np.squeeze(generated_image, axis=3)

            generated_images.append(generated_image)

            # 실제 이미지 선택
            real_image = spread_seq[i]
            real_images.append(real_image)

            total_samples_collected += 1

    except StopIteration:
        print("더 이상 가져올 데이터가 없습니다.")
        break

end_time = time.time()
during_time = end_time - start_time

# 생성된 이미지 시각화
fig, axs = plt.subplots(nrows=int(num_samples/2), ncols=4, figsize=(12, 2*int(num_samples/2)))
idx = 0
for i in range(int(num_samples/2)):
    axs[i, 0].imshow(real_images[idx][0], cmap='hot')
    axs[i, 0].axis('off')
    axs[i, 1].imshow(generated_images[idx][0], cmap='hot')
    axs[i, 1].axis('off')
    axs[i, 2].imshow(real_images[idx][1], cmap='hot')
    axs[i, 2].axis('off')
    axs[i, 3].imshow(generated_images[idx][1], cmap='hot')
    axs[i, 3].axis('off')
    idx += 2

# 간격 조정
plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0, right=1, top=1, bottom=0)
plt.show()

# MSE 및 SSIM 계산 함수
def calculate_mse(image1, image2):
    return np.mean((image1 - image2) ** 2)

def calculate_ssim(image1, image2):
    
    if isinstance(image1, tf.Tensor):   
        
        image1 = image1.numpy()
    if isinstance(image2, tf.Tensor):
        image2 = image2.numpy()
    return compare_ssim(image1, np.squeeze(image2, axis=2), data_range=1.0)

# MSE 및 SSIM 계산
mse_values = [calculate_mse(generated_images[i][1], real_images[i][1]) for i in range(num_samples)]
ssim_values = [calculate_ssim(generated_images[i][1], real_images[i][1]) for i in range(num_samples)]

# MSE 및 SSIM 값 출력 (지수 표기법 사용)
for i in range(num_samples):
    mse_formatted = np.format_float_scientific(mse_values[i], precision=2)
    ssim_formatted = np.format_float_scientific(ssim_values[i], precision=2)
    print(f'Sample {i+1} - MSE: {mse_formatted}, SSIM: {ssim_formatted}')

print(f'Average Elapsed Time: {during_time / num_samples} seconds')

In [ ]:
"""
prepare mid_size subset train data
"""


train_mid = load_subdataset("./ignition/train_mid_2step.tfrecord", batch_size=16)
fixed_sample = list(train_mid.take(1))[0]

terrain, weather_seq, spread_seq = fixed_sample
fixed_spread = tf.expand_dims(spread_seq[0],axis=0)
fixed_terrain = tf.expand_dims(terrain[0], axis=0)        # shape: (1,256,256,3)
fixed_weather = tf.expand_dims(weather_seq[0], axis=0) 
fixed_inputs = (terrain,weather_seq, spread_seq )


# terrain: (1,256,256,3)
# weather_seq: (1,3,256,256,4)
# spread_seq: (1,3,256,256,1)
terrain_np = terrain[0].numpy()  # (256,256,3)
weather_np = weather_seq[0].numpy()  # (3,256,256,4)
spread_np = spread_seq[0].numpy()  # (3,256,256,1)

# 시각화를 위한 Figure 생성
plt.figure(figsize=(16, 8))

# 1. Terrain 이미지 (RGB)
plt.subplot(2, 3, 1)
plt.imshow(terrain_np)
plt.title("Terrain (RGB)")
plt.axis("off")

# 2. Weather 정보: 각 시점별 텍스트로 표시 (온도, 습도, 풍속, 풍향)
time_labels = ["0h", "6h"]
for t in range(2):
    plt.subplot(2, 3, t+2)
    # 기상 데이터는 전체가 동일 값이므로 (256,256,4) 중 좌상단 픽셀만 사용
    weather_vals = weather_np[t, 0, 0, :]  # shape: (4,)
    text_str = (f"Temp: {weather_vals[0]:.2f}\n"
                f"Hum: {weather_vals[1]:.2f}\n"
                f"Ws: {weather_vals[2]:.2f}\n"
                f"Wd: {weather_vals[3]:.2f}")
    plt.text(0.5, 0.5, text_str, fontsize=12, ha='center', va='center', transform=plt.gca().transAxes)
    plt.title(f"Weather {time_labels[t]}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# 3. Spread 데이터: 각 시점별 산불 확산 이미지 (그레이스케일)
plt.figure(figsize=(12, 4))
spread_time_labels = [ "6h", "12h"]
for t in range(2):
    plt.subplot(1, 3, t+1)
    # spread_np[t]: (256,256,1) → squeeze해서 (256,256)
    plt.imshow(np.squeeze(spread_np[t]), cmap='gray')
    plt.title(f"Spread {spread_time_labels[t]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
"""
train model with mid_subsat train data
"""

save_pred_cb = SavePredictions(fixed_sample=fixed_inputs, output_dir="./ignition/lstm_check_mid")
epoch_tracker = EpochTracker()
cond_gan.fit(train_mid, epochs=500, callbacks=[epoch_tracker, save_pred_cb])

In [ ]:
"""
show test dataset prediction image for MID_SUBSET
"""
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import time
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim
import tensorflow as tf

# 테스트 데이터셋에서 샘플 수 설정
num_samples = 10
test_small = load_subdataset("./ignition/test_mid_2step.tfrecord", batch_size=16)
#test_small = load_subdataset("test_small.tfrecord", batch_size=16)
# test_dataset에서 데이터를 추출하기 위한 반복자 생성
test_iterator = iter(test_small)

# 생성된 이미지 저장
start_time = time.time()
generated_images = []
real_images = []

total_samples_collected = 0  # 수집된 샘플 수 추적

while total_samples_collected < num_samples:
    try:
        # test_dataset에서 하나의 배치를 가져옴
        sample_batch = next(test_iterator)
        terrain, weather_seq, spread_seq = sample_batch
    # terrain: (1,256,256,3)
    # weather_seq: (1,3,256,256,4)
    # spread_seq: (1,3,256,256,1)
        

        # 배치 내의 각 샘플을 순회하여 처리
        for i in range(terrain.shape[0]):
            if total_samples_collected >= num_samples:
                break

            # noise 생성
            fake_tensor = tf.zeros((1, 128))
            terrain_np = terrain[0].numpy()  # (256,256,3)
            weather_np = weather_seq[0].numpy()  # (2,256,256,4)
            spread_np = spread_seq[0].numpy()  # (2,256,256,1)
            # 하나의 샘플만 선택

            # generator로 이미지 생성
            generated_image = generator.predict([terrain, weather_seq])
            
            generated_image = generated_image[i]
            generated_image = np.squeeze(generated_image, axis=3)

            generated_images.append(generated_image)

            # 실제 이미지 선택
            real_image = spread_seq[i]
            real_images.append(real_image)

            total_samples_collected += 1

    except StopIteration:
        print("더 이상 가져올 데이터가 없습니다.")
        break

end_time = time.time()
during_time = end_time - start_time

# 생성된 이미지 시각화
fig, axs = plt.subplots(nrows=int(num_samples/2), ncols=4, figsize=(12, 2*int(num_samples/2)))
idx = 0
for i in range(int(num_samples/2)):
    axs[i, 0].imshow(real_images[idx][0], cmap='hot')
    axs[i, 0].axis('off')
    axs[i, 1].imshow(generated_images[idx][0], cmap='hot')
    axs[i, 1].axis('off')
    axs[i, 2].imshow(real_images[idx][1], cmap='hot')
    axs[i, 2].axis('off')
    axs[i, 3].imshow(generated_images[idx][1], cmap='hot')
    axs[i, 3].axis('off')
    idx += 2

# 간격 조정
plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0, right=1, top=1, bottom=0)
plt.show()

# MSE 및 SSIM 계산 함수
def calculate_mse(image1, image2):
    return np.mean((image1 - image2) ** 2)

def calculate_ssim(image1, image2):
    
    if isinstance(image1, tf.Tensor):
        image1 = image1.numpy()
    if isinstance(image2, tf.Tensor):
        image2 = image2.numpy()
    return compare_ssim(image1, np.squeeze(image2, axis=2), data_range=1.0)

# MSE 및 SSIM 계산
mse_values = [calculate_mse(generated_images[i][1], real_images[i][1]) for i in range(num_samples)]
ssim_values = [calculate_ssim(generated_images[i][1], real_images[i][1]) for i in range(num_samples)]

# MSE 및 SSIM 값 출력 (지수 표기법 사용)
for i in range(num_samples):
    mse_formatted = np.format_float_scientific(mse_values[i], precision=2)
    ssim_formatted = np.format_float_scientific(ssim_values[i], precision=2)
    print(f'Sample {i+1} - MSE: {mse_formatted}, SSIM: {ssim_formatted}')

print(f'Average Elapsed Time: {during_time / num_samples} seconds')

In [ ]:
# noise 생성
fake_tensor = tf.zeros((1, 128))
f,s , terrain_b, ws_b, wd_b, temp_b = check_data

generated_image = generator.predict([fake_tensor, terrain_b, ws_b, wd_b, temp_b])
generated_image = generated_image[0]
generated_image = generated_image


plt.imshow(generated_image, cmap='hot')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import time
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim
import tensorflow as tf

# 테스트 데이터셋에서 샘플 수 설정
num_samples = 100
#test_small = load_subdataset("test_small.tfrecord", batch_size=16)
# test_dataset에서 데이터를 추출하기 위한 반복자 생성
test_iterator = iter(test_small)

# 생성된 이미지 저장
start_time = time.time()
generated_images = []
real_images = []

total_samples_collected = 0  # 수집된 샘플 수 추적

while total_samples_collected < num_samples:
    try:
        # test_dataset에서 하나의 배치를 가져옴
        sample_batch = next(test_iterator)
        terrain, weather_seq, spread_seq = sample_batch
    # terrain: (1,256,256,3)
    # weather_seq: (1,3,256,256,4)
    # spread_seq: (1,3,256,256,1)
        

        # 배치 내의 각 샘플을 순회하여 처리
        for i in range(terrain.shape[0]):
            if total_samples_collected >= num_samples:
                break

            # noise 생성
            fake_tensor = tf.zeros((1, 128))
            terrain_np = terrain[0].numpy()  # (256,256,3)
            weather_np = weather_seq[0].numpy()  # (3,256,256,4)
            spread_np = spread_seq[0].numpy()  # (3,256,256,1)
            # 하나의 샘플만 선택

            # generator로 이미지 생성
            generated_image = generator.predict([terrain, weather_seq])
            
            generated_image = generated_image[i]
            generated_image = np.squeeze(generated_image, axis=3)

            generated_images.append(generated_image)

            # 실제 이미지 선택
            real_image = spread_seq[i]
            real_images.append(real_image)

            total_samples_collected += 1

    except StopIteration:
        print("더 이상 가져올 데이터가 없습니다.")
        break

end_time = time.time()
during_time = end_time - start_time

# 생성된 이미지 시각화
fig, axs = plt.subplots(nrows=int(num_samples/3), ncols=6, figsize=(12, 2*int(num_samples/3)))
idx = 0
for i in range(int(num_samples/3)):
    axs[i, 0].imshow(real_images[idx][0], cmap='hot')
    axs[i, 0].axis('off')
    axs[i, 1].imshow(generated_images[idx][0], cmap='hot')
    axs[i, 1].axis('off')
    axs[i, 2].imshow(real_images[idx][1], cmap='hot')
    axs[i, 2].axis('off')
    axs[i, 3].imshow(generated_images[idx][1], cmap='hot')
    axs[i, 3].axis('off')
    axs[i, 4].imshow(real_images[idx][2], cmap='hot')
    axs[i, 4].axis('off')
    axs[i, 5].imshow(generated_images[idx][2], cmap='hot')
    axs[i, 5].axis('off')
    idx += 3

# 간격 조정
plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0, right=1, top=1, bottom=0)
plt.show()

# MSE 및 SSIM 계산 함수
def calculate_mse(image1, image2):
    return np.mean((image1 - image2) ** 2)

def calculate_ssim(image1, image2):
    
    if isinstance(image1, tf.Tensor):
        image1 = image1.numpy()
    if isinstance(image2, tf.Tensor):
        image2 = image2.numpy()
    return compare_ssim(image1, np.squeeze(image2, axis=2), data_range=1.0)

# MSE 및 SSIM 계산
mse_values = [calculate_mse(generated_images[i][2], real_images[i][2]) for i in range(num_samples)]
ssim_values = [calculate_ssim(generated_images[i][2], real_images[i][2]) for i in range(num_samples)]

# MSE 및 SSIM 값 출력 (지수 표기법 사용)
for i in range(num_samples):
    mse_formatted = np.format_float_scientific(mse_values[i], precision=2)
    ssim_formatted = np.format_float_scientific(ssim_values[i], precision=2)
    print(f'Sample {i+1} - MSE: {mse_formatted}, SSIM: {ssim_formatted}')

print(f'Average Elapsed Time: {during_time / num_samples} seconds')




In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import time
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim
import tensorflow as tf

# 테스트 데이터셋에서 샘플 수 설정
num_samples = 15

# test_dataset에서 데이터를 추출하기 위한 반복자 생성
test_iterator = iter(test_dataset2)

# 생성된 이미지 저장
start_time = time.time()
generated_images = []
real_images = []

total_samples_collected = 0  # 수집된 샘플 수 추적

while total_samples_collected < num_samples:
    try:
        # test_dataset에서 하나의 배치를 가져옴
        sample_batch = next(test_iterator)
        sample_perimeter, sample_terrains, sample_wind_speeds, sample_wind_directions, sample_temp = sample_batch
        # 배치 내의 각 샘플을 순회하여 처리
        for i in range(sample_terrains.shape[0]):
            if total_samples_collected >= num_samples:
                break

            # noise 생성
            fake_tensor = tf.zeros((1, 128))

            # 하나의 샘플만 선택
            sample_terrain = sample_terrains[i:i+1]
            sample_wind_speed = sample_wind_speeds[i:i+1]
            sample_wind_direction = sample_wind_directions[i:i+1]
            sample_temp_single = sample_temp[i:i+1]
            # generator로 이미지 생성
            generated_image = generator.predict([fake_tensor, sample_terrain, sample_wind_speed, sample_wind_direction, sample_temp_single])
            generated_image = generated_image[0]
            generated_image = generated_image
            generated_image = np.squeeze(generated_image, axis=2)

            generated_images.append(generated_image)

            # 실제 이미지 선택
            real_image = sample_perimeter[i] * 0.5 + 0.5
            real_images.append(real_image)

            total_samples_collected += 1

    except StopIteration:
        print("더 이상 가져올 데이터가 없습니다.")
        break

end_time = time.time()
during_time = end_time - start_time

# 생성된 이미지 시각화
fig, axs = plt.subplots(nrows=int(num_samples/3), ncols=6, figsize=(12, 2*int(num_samples/3)))
idx = 0
for i in range(int(num_samples/3)):
    axs[i, 0].imshow(real_images[idx], cmap='hot')
    axs[i, 0].axis('off')
    axs[i, 1].imshow(generated_images[idx], cmap='hot')
    axs[i, 1].axis('off')
    axs[i, 2].imshow(real_images[idx+1], cmap='hot')
    axs[i, 2].axis('off')
    axs[i, 3].imshow(generated_images[idx+1], cmap='hot')
    axs[i, 3].axis('off')
    axs[i, 4].imshow(real_images[idx+2], cmap='hot')
    axs[i, 4].axis('off')
    axs[i, 5].imshow(generated_images[idx+2], cmap='hot')
    axs[i, 5].axis('off')
    idx += 3

# 간격 조정
plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0, right=1, top=1, bottom=0)
plt.show()

# MSE 및 SSIM 계산 함수
def calculate_mse(image1, image2):
    return np.mean((image1 - image2) ** 2)

def calculate_ssim(image1, image2):
    
    if isinstance(image1, tf.Tensor):
        image1 = image1.numpy()
    if isinstance(image2, tf.Tensor):
        image2 = image2.numpy()
    return compare_ssim(image1, np.squeeze(image2, axis=2), data_range=1.0)

# MSE 및 SSIM 계산
mse_values = [calculate_mse(generated_images[i], real_images[i]) for i in range(num_samples)]
ssim_values = [calculate_ssim(generated_images[i], real_images[i]) for i in range(num_samples)]

# MSE 및 SSIM 값 출력 (지수 표기법 사용)
for i in range(num_samples):
    mse_formatted = np.format_float_scientific(mse_values[i], precision=2)
    ssim_formatted = np.format_float_scientific(ssim_values[i], precision=2)
    print(f'Sample {i+1} - MSE: {mse_formatted}, SSIM: {ssim_formatted}')

print(f'Average Elapsed Time: {during_time / num_samples} seconds')




In [ ]:
g_idx =13
print(generated_images[g_idx])
# 1. NumPy 배열로 이미지 생성 (예: 흑백 이미지)
# 흑백 이미지 (값 범위: 0~255)

# 2. 컬러맵 적용 후 단독 이미지 저장
plt.imshow(generated_images[g_idx], cmap='hot')  # 'hot' 컬러맵 사용
plt.axis('off')  # 축 제거
plt.savefig("./compare_result/gan.png", bbox_inches='tight', pad_inches=0)
plt.close()  # 플롯 닫기
print("이미지가 저장되었습니다: colormap_image.png")

In [ ]:
real = perimeter[0]

plt.imshow(real, cmap='gray')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import time
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim

# 예제에서 사용할 샘플 수 설정
num_samples = 12

# 샘플 인덱스를 랜덤으로 선택
indices = np.random.randint(0, 1300, size=num_samples)

# 생성된 이미지 저장
start_time = time.time()
generated_images = []
real_images = []

for ci in indices:
    noise = gene_rdnormal(1)
    fake_tensor = tf.zeros((1, 128))
    
    sample_terrains = tf.expand_dims(terrain_images[ci], axis=0)
    sample_wind_speeds = tf.expand_dims(wind_speed[ci], axis=0)
    sample_wind_directions = tf.expand_dims(wind_direction[ci], axis=0)
    sample_temp = tf.expand_dims(temp[ci], axis=0)

    generated_image = generator.predict([fake_tensor, sample_terrains, sample_wind_speeds, sample_wind_directions, sample_temp])
    generated_image = generated_image[0]
    generated_image = Lslope5 * generated_image + 0.5
    generated_image = np.squeeze(generated_image, axis=2)
    
    generated_images.append(generated_image)

    real_image = perimeter[ci:ci+1][0] * 0.25 + 0.5
    real_images.append(real_image)

end_time = time.time()
during_time = end_time - start_time

# 생성된 이미지 시각화
fig, axs = plt.subplots(nrows=int(num_samples/2), ncols=4, figsize=(8, 2 * int(num_samples/2)))
idx = 0
for i in range(int(num_samples/2)):
    
    axs[i, 0].imshow(real_images[idx], cmap='hot')
    axs[i, 0].axis('off')
    axs[i, 1].imshow(generated_images[idx], cmap='hot')
    axs[i, 1].axis('off')
    axs[i, 2].imshow(real_images[idx+1], cmap='hot')
    axs[i, 2].axis('off')
    axs[i, 3].imshow(generated_images[idx+1], cmap='hot')
    axs[i, 3].axis('off')
    idx += 2


# 간격 조정
plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0, right=1, top=1, bottom=0)

plt.show()

def calculate_mse(image1, image2):
    return np.mean((image1 - image2) ** 2)

def calculate_ssim(image1, image2):
    return compare_ssim(image1, image2, data_range=1.0)

# MSE 및 SSIM 계산
mse_values = [calculate_mse(generated_images[i], real_images[i]) for i in range(num_samples)]
ssim_values = [calculate_ssim(generated_images[i], real_images[i]) for i in range(num_samples)]
# MSE 및 SSIM 값 출력 (지수 표기법 사용)
for i in range(num_samples):
    mse_formatted = np.format_float_scientific(mse_values[i], precision=2)
    ssim_formatted = np.format_float_scientific(ssim_values[i], precision=2)
    print(f'Sample {i+1} - MSE: {mse_formatted}, SSIM: {ssim_formatted}')

print(f'Average Elapsed Time: {during_time / num_samples} seconds')



In [ ]:
print(ci)

In [ ]:
########################################
#Save gene, disc model with HDF5 format#
########################################

generator.save("ch3_gray_gene.h5")
discriminator.save("ch3_gray_disc.h5")

In [ ]:
generator.load_weights('ch3_gray_gene.h5')

In [ ]:
def calculate_mse(img1, img2):
    return np.mean((img1 - img2) ** 2)

def calculate_ssim(image1, image2):
     # Tensor를 NumPy 배열로 변환
    print(image1.shape)
    print(image2.shape)
    if isinstance(image1, tf.Tensor):
        image1 = image1.numpy()
    if isinstance(image2, tf.Tensor):
        image2 = image2.numpy()
    return compare_ssim(np.squeeze(image1, axis=2), image2, data_range=1.0)

def evaluate_model(generator, dataset):
    max_mse = 0
    min_ssim = float('inf')  # SSIM의 최소값을 찾기 위해 초기값을 무한대로 설정
    min_mse = float('inf')  # MSE의 최소값을 찾기 위해 초기값을 무한대로 설정

    max_mse_case = None
    min_mse_case = None
    min_ssim_case = None
    total_mse = 0
    total_ssim = 0
    count = 0
    mse_all = []
    for batch in dataset:
        perimeter, terrain, ws, wd, temp = batch

        # 배치 크기에 맞는 fake_tensor 생성
        batch_size = perimeter.shape[0]
        fake_tensor = tf.zeros((batch_size, 128))

        # Generator 예측 (perimeter 자리에 fake_tensor 사용)
        try:
            prediction = generator.predict([fake_tensor, terrain, ws, wd, temp])
        except Exception as e:
            print(f"Error during prediction: {e}")
            break

        for i in range(batch_size):
            real_image = perimeter[i] * 0.5 + 0.5  # 실제 이미지 (실제 Y 값)
            pred_image = prediction[i] * 0.5+ 0.5  # 예측된 이미지 (Generator의 출력)
            pred_image = np.squeeze(pred_image, axis=2)

            mse_value = calculate_mse(real_image, pred_image)
            ssim_value = calculate_ssim(real_image, pred_image)
            mse_all.append(mse_value)
            total_mse += mse_value
            total_ssim += ssim_value
            count += 1

            # 최대 MSE 확인
            if mse_value > max_mse:
                max_mse = mse_value
                max_mse_case = (real_image, pred_image)
            
            # 최소 MSE 확인
            if mse_value < min_mse:
                min_mse = mse_value
                min_mse_case = (real_image, pred_image)


            # 최소 SSIM 확인
            if ssim_value < min_ssim:
                min_ssim = ssim_value
                min_ssim_case = (real_image, pred_image)

    avg_mse = total_mse / count
    avg_ssim = total_ssim / count

    return max_mse, min_ssim, avg_mse, avg_ssim, max_mse_case, min_ssim_case, min_mse ,min_mse_case, mse_all 


def plot_comparison(real_image, pred_image, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(real_image, cmap='hot')
    axes[0].set_title('Real Image')
    axes[1].imshow(pred_image, cmap='hot')
    axes[1].set_title('Predicted Image')
    plt.suptitle(title)
    plt.show()



In [ ]:
# Train 데이터셋에 대한 평가
train_max_mse, train_min_ssim, train_avg_mse, train_avg_ssim, train_max_mse_case, train_min_ssim_case, train_min_mse, train_min_mse_case, train_mse_all = evaluate_model(generator, train_dataset)

# Test 데이터셋에 대한 평가
test_max_mse, test_min_ssim, test_avg_mse, test_avg_ssim, test_max_mse_case, test_min_ssim_case, test_min_mse, test_min_mse_case, test_mse_all = evaluate_model(generator, test_dataset)





In [ ]:
# 결과 출력
print(f"Train Dataset: Avg MSE = {train_avg_mse}, Avg SSIM = {train_avg_ssim}, Max MSE={train_max_mse}")
print(f"Test Dataset: Avg MSE = {test_avg_mse}, Avg SSIM = {test_avg_ssim}, Max MSE={test_max_mse}")

# 최대 MSE 케이스 시각화
real_image, pred_image = train_max_mse_case
plot_comparison(real_image, pred_image, "Train Dataset - Max MSE Case")

real_image, pred_image = test_max_mse_case
plot_comparison(real_image, pred_image, "Test Dataset - Max MSE Case")


# 최대 SSIM 케이스 시각화
real_image, pred_image = train_min_mse_case
plot_comparison(real_image, pred_image, "Train Dataset - Min MSE Case")

real_image, pred_image = test_min_mse_case
plot_comparison(real_image, pred_image, "Test Dataset - Min MSE Case")

# 최대 SSIM 케이스 시각화
#real_image, pred_image = train_min_ssim_case
#plot_comparison(real_image, pred_image, "Train Dataset - Min SSIM Case")

#real_image, pred_image = test_min_ssim_case
#plot_comparison(real_image, pred_image, "Test Dataset - Min SSIM Case")


In [ ]:
# 특정 값 설정
threshold = 0.0015

# threshold보다 큰 값의 비율을 계산
train_above = (np.sum(np.array(train_mse_all) > threshold) / len(train_mse_all)) * 100
test_above = (np.sum(np.array(test_mse_all) > threshold) / len(test_mse_all)) * 100
print(train_above)
print(test_above)

# 데이터를 정렬
sorted_data1 = np.sort(train_mse_all)
sorted_data2 = np.sort(test_mse_all)

# 데이터의 누적 비율 계산
cumulative_percent1 = np.arange(1, len(sorted_data1) + 1) / len(sorted_data1) * 100
cumulative_percent2 = np.arange(1, len(sorted_data2) + 1) / len(sorted_data2) * 100

# 그래프 그리기
plt.figure(figsize=(8, 6))
plt.plot(sorted_data1, cumulative_percent1, label="MSE for train_data")
plt.plot(sorted_data2, cumulative_percent2, label="MSE for test_data")
plt.axhline(y=100, color='r', linestyle='--', label="100% Line")
plt.xlim(0,0.04)
plt.title('MSE between real and predicted(CDF)')
plt.xlabel('Value')
plt.ylabel('Percentage(%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(train_max_mse, train_min_ssim)
print(test_max_mse, test_min_ssim)